# TP3 – Recomendador personalizado de libros

**UNR · TUIA · Procesamiento de Lenguaje Natural – Unidad 3**

**Integrantes:** Franco Renna · Micaela Gómez · Irma Carolina Ruiz · Franco Esparza

El TP1 dejó un catálogo de libros y el TP2 representó sus sinopsis para buscar libros
parecidos. Este trabajo incorpora las preferencias de un lector para recomendarle libros.

Antes hay un problema: el catálogo ampliado que entrega la cátedra está **contaminado**. En
algunas sinopsis se insertaron enlaces, mensajes promocionales y hasta libros que no existen,
para que el recomendador muestre esos sitios. Por eso el trabajo tiene dos mitades: primero
se audita el catálogo (pasos 1 y 2) y después se construye el recomendador (pasos 3 a 8).

Las instrucciones para instalar el entorno y ejecutar este notebook están en el `README.md`
del repositorio.

## Librerías y archivos

Para el Paso 1 alcanzan tres cosas: `pandas` para manejar la tabla de libros, `re` y
`unicodedata` para normalizar texto, y `thefuzz` para medir la semejanza entre cadenas.
`thefuzz` implementa la distancia de Levenshtein y algunas variantes que comparan los títulos
palabra por palabra en lugar de carácter por carácter.

In [1]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from thefuzz import fuzz

# El catalogo que entrega la catedra y la carpeta donde guardamos resultados intermedios.
RUTA_CATALOGO = Path("catalogo_tp3.jsonl")
CARPETA_SALIDAS = Path("salidas")
CARPETA_SALIDAS.mkdir(exist_ok=True)

# Para que las sinopsis largas no se corten al mostrarlas.
pd.set_option("display.max_colwidth", 120)

## Carga del catálogo

El archivo viene en formato JSON Lines: un libro por línea. De los doce campos que trae nos
quedamos con seis.

| Campo | Para qué lo usamos |
| --- | --- |
| `slug` | identificador del libro, como pide la consigna |
| `title` | para mostrar el título |
| `authors` | entrada del Paso 1: solo comparamos libros del mismo autor |
| `synopsis` | el texto con el que trabaja todo el TP |
| `genres`, `series` | no los pide el Paso 1, pero ayudan en el Paso 2 a juzgar si un libro es inventado |

In [2]:
catalogo = pd.read_json(RUTA_CATALOGO, lines=True)

print(f"Libros en el archivo: {len(catalogo)}")
print(f"Campos: {list(catalogo.columns)}")

Libros en el archivo: 8747
Campos: ['source_url', 'slug', 'title', 'authors', 'genres', 'seed_genre', 'series', 'series_number', 'cover_url', 'synopsis', 'synopsis_chars', 'scraped_at']


In [3]:
libros = catalogo[["slug", "title", "authors", "genres", "series", "synopsis"]].copy()

# Vamos anotando cuantos libros elimina cada motivo, para el resumen final.
eliminados = {}

libros.head(5)

,slug,title,authors,genres,series,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",NaN,"La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos...."
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, Ensayo]",NaN,La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importantes controversias sobre la pr...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo, Espiritualidad]",NaN,"Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de nuestra era, Teotihuacan dejar..."
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Política, Viajes]",NaN,"Entre los siglos VIII y VI a. C. los fenicios establecieron el primer sistema comercial mediterráneo, que abarcó des..."
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",NaN,Las raíces evolutivas del sufrimiento humano\nUna nueva comprensión de la salud mental que brinda una hipótesis reve...


**Resultado.** El catálogo trae **8747 libros** y doce campos. Nos quedamos con seis y abrimos
el diccionario `eliminados`, donde anotamos cuántos libros saca cada criterio: la consigna
pide informar al final cuántos se eliminaron **por cada motivo**.

## Registros sin sinopsis

Un libro sin sinopsis no sirve para nada de lo que viene: todo el TP trabaja sobre ese texto.
Consideramos que falta la sinopsis tanto si el campo es nulo como si es una cadena vacía o
solo espacios.

In [4]:
sin_sinopsis = libros["synopsis"].isna() | (libros["synopsis"].astype(str).str.strip() == "")

print(f"Libros sin sinopsis: {sin_sinopsis.sum()}")
libros[sin_sinopsis][["slug", "title", "synopsis"]]

Libros sin sinopsis: 1


,slug,title,synopsis
7969,calvario,Calvario,NaN


In [5]:
eliminados["sin sinopsis"] = int(sin_sinopsis.sum())
libros = libros[~sin_sinopsis]

print(f"Quedan {len(libros)} libros con sinopsis.")

Quedan 8746 libros con sinopsis.


**Resultado.** Hay **un solo libro** sin sinopsis. Lo eliminamos y quedan 8746.

## Duplicados por identificador

El `slug` es el identificador del libro. Si el mismo `slug` apareciera dos veces, tendríamos
el mismo registro repetido y habría que conservar uno.

In [6]:
slug_repetido = libros["slug"].duplicated()

print(f"Identificadores repetidos: {slug_repetido.sum()}")

eliminados["slug duplicado"] = int(slug_repetido.sum())
libros = libros[~slug_repetido]

Identificadores repetidos: 0


**Resultado.** No hay ningún `slug` repetido, así que este criterio no elimina nada. Dejamos
el chequeo escrito igual: si la cátedra actualizara el catálogo, el notebook lo detectaría.

## Normalización de los espacios

Al descargar las páginas, las palabras en cursiva quedaron separadas por saltos de línea. Por
ejemplo, una sinopsis puede contener `"El primer americano\n, ambiciosa obra..."`. Eso rompe
las palabras y, más adelante, haría que dos sinopsis iguales parezcan distintas.

Reemplazamos cada secuencia de espacios en blanco —espacios, tabulaciones y saltos de línea—
por un solo espacio, y recortamos los extremos.

**Importante:** por ahora **no** eliminamos ningún otro carácter. Varios de los caracteres
raros que trae el texto son justamente la evidencia que vamos a necesitar en el Paso 2
(caracteres invisibles metidos dentro de un enlace, letras de otro alfabeto que se ven
latinas). Si los borráramos ahora, perderíamos la pista.

Normalizamos solo `title` y `synopsis`: el `slug` no tiene espacios.

In [7]:
columnas_texto = ["title", "synopsis"]

# Filas donde hay espacios repetidos o saltos de linea.
filas_a_normalizar = libros[
    libros[columnas_texto].apply(lambda col: col.str.contains(r"\s{2,}|\n", na=False)).any(axis=1)
]
print(f"Filas con espacios repetidos o saltos de linea: {len(filas_a_normalizar)}")

filas_a_normalizar[["title", "synopsis"]].head(3)

Filas con espacios repetidos o saltos de linea: 5253


,title,synopsis
0,De la caverna al cosmos,"La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos...."
1,Quiénes somos y cómo hemos llegado hasta aquí,La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importantes controversias sobre la pr...
2,Teotihuacan,"Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de nuestra era, Teotihuacan dejar..."


In [8]:
for columna in columnas_texto:
    libros[columna] = libros[columna].str.replace(r"\s+", " ", regex=True).str.strip()

print("Espacios normalizados.")
libros[["title", "synopsis"]].head(3)

Espacios normalizados.


,title,synopsis
0,De la caverna al cosmos,"La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos...."
1,Quiénes somos y cómo hemos llegado hasta aquí,La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importantes controversias sobre la pr...
2,Teotihuacan,"Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de nuestra era, Teotihuacan dejar..."


**Resultado.** **5253 libros** —más de la mitad del catálogo— tenían saltos de línea o
espacios repetidos dentro del título o la sinopsis. Después de normalizar, cada texto es una
sola línea con espacios simples.

Este paso va **antes** de buscar sinopsis repetidas: dos sinopsis que solo se diferenciaban en
un salto de línea recién ahora son exactamente iguales.

## Sinopsis exactamente repetidas

La consigna avisa de revisar los duplicados antes de borrarlos, porque en este catálogo hay
cursos y series completas que repiten la misma sinopsis en todos sus volúmenes. Primero
miramos qué son esos grupos y después conservamos un libro por sinopsis.

In [9]:
conteo = libros.groupby("synopsis").size().sort_values(ascending=False)
repetidas = conteo[conteo > 1]

print(f"Sinopsis distintas que aparecen mas de una vez: {len(repetidas)}")
print(f"Libros involucrados: {repetidas.sum()}")
print(f"Grupo mas grande: {repetidas.iloc[0]} libros")

# Los titulos del grupo mas grande, para ver de que se trata.
libros[libros["synopsis"] == repetidas.index[0]][["slug", "title"]].head(6)

Sinopsis distintas que aparecen mas de una vez: 33
Libros involucrados: 131
Grupo mas grande: 58 libros


,slug,title
2671,aleman-unidad-40,Alemán – Unidad 40
2680,aleman-unidad-15,Alemán – Unidad 15
2681,aleman-unidad-13,Alemán – Unidad 13
2682,aleman-unidad-14,Alemán – Unidad 14
4086,aleman-unidad-34,Alemán – Unidad 34
4087,aleman-unidad-33,Alemán – Unidad 33


In [10]:
antes = len(libros)
libros = libros.drop_duplicates(subset=["synopsis"], keep="first")

eliminados["sinopsis duplicada"] = antes - len(libros)
print(f"Eliminados por sinopsis repetida: {eliminados['sinopsis duplicada']}")
print(f"Quedan {len(libros)} libros.")

Eliminados por sinopsis repetida: 98
Quedan 8648 libros.


**Resultado.** Hay **33 sinopsis** que se repiten y entre todas abarcan **131 libros**. El
grupo más grande tiene **58 libros**: son las unidades didácticas de un curso de alemán, que
comparten la misma descripción del curso. Son libros distintos, pero su texto no aporta nada
diferente, así que conservamos uno por sinopsis y eliminamos **98**.

Es la primera decisión discutible del paso: con sinopsis idénticas el recomendador no tiene
forma de distinguir esos 58 volúmenes entre sí, y dejarlos todos haría que una sola
descripción pesara 58 veces en el catálogo.

## Casi duplicados: la misma obra con otro título

Falta un caso más difícil: la misma obra cargada dos veces con títulos **parecidos pero no
iguales**, por ejemplo con y sin subtítulo, o en dos traducciones distintas.

Para no comparar los 8648 libros todos contra todos —serían casi 37 millones de pares—
comparamos **solo libros del mismo autor**, como sugiere la consigna.

El problema es que la semejanza de cadenas mide la **forma**, no el **significado**:
`"La Perla número 8"` y `"La Perla número 5"` se parecen en un 94 % y son libros distintos.
Nuestro criterio tiene tres partes:

1. **Los títulos se parecen** (`token_sort_ratio >= 90`), comparando el título *sin* los
   paréntesis de edición: `"Hamlet (Bilingüe)"` y `"Hamlet"` son la misma obra.
2. **Veto por número de volumen.** Si los títulos contienen numerales, ordinales o números
   romanos **distintos**, son volúmenes distintos de una serie y no se unen, por más que los
   títulos se parezcan. Esto separa `número 5` de `número 8`, `tomo I` de `tomo II` y
   `siglo XIX` de `siglo XX`.
3. **Se unen** si los títulos son prácticamente iguales (`>= 98`) o si, además de parecerse,
   **las sinopsis también se parecen** (`> 90`).

La tercera parte es la que combina título y sinopsis. El caso de los títulos casi idénticos se
une sin mirar la sinopsis porque dos ediciones de la misma obra suelen traer reseñas escritas
por editoriales distintas, que no se parecen en nada entre sí.

In [11]:
# Numeros romanos del I al XXXIX, suficiente para volumenes y siglos.
ROMANO = re.compile(r"^(?:x{0,3})(?:ix|iv|v?i{0,3})$")

ORDINALES = {"primero": 1, "primera": 1, "segundo": 2, "segunda": 2,
             "tercero": 3, "tercera": 3, "cuarto": 4, "cuarta": 4,
             "quinto": 5, "quinta": 5, "sexto": 6, "sexta": 6,
             "septimo": 7, "septima": 7, "octavo": 8, "octava": 8}


def quitar_tildes(texto):
    """Pasa 'Andalucia' y 'Andalucia' con tilde a la misma forma."""
    return "".join(c for c in unicodedata.normalize("NFD", texto)
                   if unicodedata.category(c) != "Mn")


def marcas_de_volumen(titulo):
    """Numerales, ordinales y romanos que aparecen en el titulo.

    Si dos titulos tienen marcas distintas, son volumenes distintos de una serie.
    """
    marcas = set()
    for palabra in re.findall(r"[a-z0-9]+", quitar_tildes(titulo.lower())):
        if palabra.isdigit():
            marcas.add(int(palabra))
        elif palabra in ORDINALES:
            marcas.add(ORDINALES[palabra])
        elif ROMANO.fullmatch(palabra):
            marcas.add("R" + palabra)
    return marcas


def titulo_base(titulo):
    """Titulo sin los parentesis de edicion, para comparar la obra y no la edicion."""
    sin_parentesis = re.sub(r"\([^)]*\)", "", quitar_tildes(titulo.lower()))
    return re.sub(r"\s+", " ", sin_parentesis).strip()


# Comprobamos las dos funciones con los ejemplos que nombra la consigna.
ejemplos = [("La Perla numero 8", "La Perla numero 5"),
            ("Conversaciones con Dios I", "Conversaciones con Dios II"),
            ("Hamlet (Bilingue)", "Hamlet")]

for uno, otro in ejemplos:
    parecido = fuzz.token_sort_ratio(titulo_base(uno), titulo_base(otro))
    iguales = marcas_de_volumen(uno) == marcas_de_volumen(otro)
    print(f"{uno!r} vs {otro!r}")
    print(f"   titulo_base -> {titulo_base(uno)!r} / {titulo_base(otro)!r}   parecido {parecido}")
    print(f"   marcas      -> {marcas_de_volumen(uno)} / {marcas_de_volumen(otro)}"
          f"   {'iguales' if iguales else 'DISTINTAS: se separan'}")
    print()

'La Perla numero 8' vs 'La Perla numero 5'
   titulo_base -> 'la perla numero 8' / 'la perla numero 5'   parecido 94
   marcas      -> {8} / {5}   DISTINTAS: se separan

'Conversaciones con Dios I' vs 'Conversaciones con Dios II'
   titulo_base -> 'conversaciones con dios i' / 'conversaciones con dios ii'   parecido 98
   marcas      -> {'Ri'} / {'Rii'}   DISTINTAS: se separan

'Hamlet (Bilingue)' vs 'Hamlet'
   titulo_base -> 'hamlet' / 'hamlet'   parecido 100
   marcas      -> set() / set()   iguales



In [12]:
TITULO_PARECIDO = 90    # minimo para siquiera considerar dos libros
TITULO_IGUAL = 98       # titulos practicamente identicos: alcanza para unir
SINOPSIS_PARECIDA = 90  # si el titulo solo se parece, pedimos que la sinopsis tambien

# Los autores vienen como lista; los pasamos a texto para poder agrupar.
libros["autor"] = libros["authors"].apply(
    lambda a: ", ".join(a) if isinstance(a, list) else str(a)
)

a_eliminar = set()
pares_unidos = []
pares_separados = []

for autor, grupo in libros.groupby("autor"):
    if len(grupo) < 2 or autor in ("", "nan", "None"):
        continue

    indices = grupo.index.tolist()
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            uno, otro = indices[i], indices[j]
            if uno in a_eliminar or otro in a_eliminar:
                continue

            titulo_1, titulo_2 = grupo.at[uno, "title"], grupo.at[otro, "title"]

            # 1. Los titulos tienen que parecerse.
            sim_titulo = fuzz.token_sort_ratio(titulo_base(titulo_1), titulo_base(titulo_2))
            if sim_titulo < TITULO_PARECIDO:
                continue

            # 2. Veto: numeros de volumen distintos -> libros distintos.
            if marcas_de_volumen(titulo_1) != marcas_de_volumen(titulo_2):
                pares_separados.append((titulo_1, titulo_2, sim_titulo, "numeral distinto"))
                continue

            # 3. Se unen si el titulo es casi identico o si la sinopsis acompaña.
            sim_sinopsis = fuzz.token_sort_ratio(grupo.at[uno, "synopsis"],
                                                 grupo.at[otro, "synopsis"])
            if sim_titulo >= TITULO_IGUAL or sim_sinopsis > SINOPSIS_PARECIDA:
                a_eliminar.add(otro)
                pares_unidos.append((titulo_1, titulo_2, sim_titulo, sim_sinopsis))
            else:
                pares_separados.append((titulo_1, titulo_2, sim_titulo, f"sinopsis {sim_sinopsis}"))

libros = libros.drop(index=list(a_eliminar)).drop(columns=["autor"])
eliminados["casi duplicado"] = len(a_eliminar)

vetados = [p for p in pares_separados if p[3] == "numeral distinto"]
por_sinopsis = [p for p in pares_separados if p[3] != "numeral distinto"]

print(f"Pares unidos (misma obra):     {len(pares_unidos)}")
print(f"Pares separados por numeral:   {len(vetados)}")
print(f"Pares separados por sinopsis:  {len(por_sinopsis)}")
print(f"Libros eliminados:             {len(a_eliminar)}")

Pares unidos (misma obra):     56
Pares separados por numeral:   119
Pares separados por sinopsis:  6
Libros eliminados:             56


In [13]:
print("DOS PARES QUE DECIDIMOS UNIR (la misma obra)")
print()
for titulo_1, titulo_2, sim_t, sim_s in pares_unidos[:2]:
    print(f"  {titulo_1!r}")
    print(f"  {titulo_2!r}")
    print(f"  titulo {sim_t} / sinopsis {sim_s}  ->  misma obra, se conserva una")
    print()

print()
print("DOS PARES QUE DECIDIMOS SEPARAR (libros distintos)")
print()

# Mostramos pares de obras distintas, para no repetir dos veces la misma serie.
obras_vistas = set()
for titulo_1, titulo_2, sim_t, motivo in vetados:
    if titulo_base(titulo_1) in obras_vistas:
        continue
    obras_vistas.add(titulo_base(titulo_1))
    print(f"  {titulo_1!r}")
    print(f"  {titulo_2!r}")
    print(f"  titulo {sim_t}, pero {motivo}  ->  libros distintos, se conservan los dos")
    print()
    if len(obras_vistas) == 2:
        break

DOS PARES QUE DECIDIMOS UNIR (la misma obra)

  'Nuevos cuentos de Bustos Domecq'
  'Nuevos cuentos de Bustos Domecq'
  titulo 100 / sinopsis 55  ->  misma obra, se conserva una

  'El libro de los espíritus (trad. Benjamín Briggent)'
  'El libro de los espíritus (trad. Gustavo N. Martínez)'
  titulo 100 / sinopsis 64  ->  misma obra, se conserva una


DOS PARES QUE DECIDIMOS SEPARAR (libros distintos)

  'El conde de Montecristo (tomo II) (Ed. B. Oro)'
  'El conde de Montecristo (tomo I) (Ed. B. Oro)'
  titulo 100, pero numeral distinto  ->  libros distintos, se conservan los dos

  'La Perla número 8'
  'La Perla número 5'
  titulo 94, pero numeral distinto  ->  libros distintos, se conservan los dos



In [14]:
# Comprobacion del caso que el criterio anterior resolvia mal: los tres volumenes
# de "La Perla" se parecen mucho entre si y tienen que seguir estando los tres.
perla = libros[libros["title"].str.startswith("La Perla numero", na=False)
               | libros["title"].str.startswith("La Perla número", na=False)]

print(f"Volumenes de 'La Perla' en el catalogo: {len(perla)}")
print(perla["title"].tolist())

Volumenes de 'La Perla' en el catalogo: 3
['La Perla número 8', 'La Perla número 5', 'La Perla número 6']


In [15]:
# Los pares que el titulo acerco pero la sinopsis separo: son los casos mas delicados.
print("Pares con titulo parecido que la sinopsis separo")
print()
for titulo_1, titulo_2, sim_t, motivo in por_sinopsis:
    print(f"  titulo {sim_t}, {motivo}")
    print(f"    {titulo_1!r}")
    print(f"    {titulo_2!r}")

Pares con titulo parecido que la sinopsis separo

  titulo 91, sinopsis 57
    'Historias negras del fútbol argentino'
    'Nuevas historias negras del fútbol argentino'
  titulo 90, sinopsis 58
    'Divina Comedia (trad. A. Echeverría)'
    'La Divina Comedia (trad. Bartolomé Mitre)'
  titulo 90, sinopsis 58
    'Divina Comedia (trad. A. Echeverría)'
    'La divina comedia (Ilustrado)'
  titulo 94, sinopsis 63
    'Exploraciones secretas en África'
    'Exploraciones secretas en Asia'
  titulo 90, sinopsis 51
    'Fragmentos de la comedia nueva'
    'Fragmentos de la comedia media'
  titulo 92, sinopsis 72
    'Diccionario de autores, obras y personajes de la literatura Griega'
    'Diccionario de autores, obras y personajes de la literatura Latina'


**Resultado.** El criterio une **56 pares** y elimina esos 56 libros repetidos. Los pares
unidos son todos ediciones o traducciones distintas de la misma obra: *Hamlet* con y sin
edición bilingüe, el *Lazarillo de Tormes* en tres ediciones, *Las flores del mal* en tres
traducciones, *El etrusco* cargado dos veces con reseñas distintas.

El **veto por número de volumen separó 119 pares**, y ahí está lo que el criterio anterior
hacía mal: sin el veto, *La Perla número 8* se unía con la 5 y con la 6, los volúmenes de la
*Historia de la decadencia y caída del Imperio Romano* colapsaban en uno, y lo mismo los dos
tomos de *El conde de Montecristo*. Son libros distintos y todos siguen en el catálogo.

Solo **6 pares** quedaron separados por la sinopsis, y revisarlos muestra que la decisión fue
correcta: el *Diccionario de autores, obras y personajes de la literatura Griega* y el de la
*Latina* comparten el 92 % del título y son obras distintas, igual que *Exploraciones
secretas en África* y *en Asia*, o *Fragmentos de la comedia nueva* y *de la comedia media*.
El título solo no alcanzaba y la sinopsis los separó bien.

**Caso de falla.** El criterio no es perfecto. *Divina Comedia (trad. A. Echeverría)* y
*La Divina Comedia (trad. Bartolomé Mitre)* son la misma obra, pero el artículo inicial baja
la semejanza del título a 90 —justo por debajo del umbral de "casi idénticos"— y las dos
reseñas editoriales no se parecen entre sí. Quedan como dos libros y deberían ser uno.

Es el precio de usar un umbral fijo: bajarlo para capturar este caso uniría también el
*Diccionario* griego con el latino, que es un error peor. Preferimos equivocarnos dejando un
duplicado antes que borrando un libro legítimo.

## Resumen del Paso 1

La consigna pide cerrar el paso mostrando cinco registros, cuántos libros quedaron disponibles
y cuántos se eliminaron por cada motivo.

In [16]:
resumen = pd.DataFrame(
    [{"motivo": motivo, "libros eliminados": cantidad}
     for motivo, cantidad in eliminados.items()]
)
resumen.loc[len(resumen)] = {"motivo": "TOTAL", "libros eliminados": sum(eliminados.values())}

print(f"Libros en el catalogo original: {len(catalogo)}")
print(f"Libros disponibles al terminar: {len(libros)}")
print()
resumen

Libros en el catalogo original: 8747
Libros disponibles al terminar: 8592



,motivo,libros eliminados
0,sin sinopsis,1
1,slug duplicado,0
2,sinopsis duplicada,98
3,casi duplicado,56
4,TOTAL,155


In [17]:
libros.head(5)

,slug,title,authors,genres,series,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",NaN,"La Humanidad ha entrado en una nueva fase evolutiva. Esto implica cambios cada vez más rápidos, panoramas inciertos...."
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, Ensayo]",NaN,La extracción de ADN de los huesos de nuestros ancestros ha permitido resolver importantes controversias sobre la pr...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo, Espiritualidad]",NaN,"Teotihuacan es la primera gran ciudad del centro de México. Surgida en los albores de nuestra era, Teotihuacan dejar..."
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Política, Viajes]",NaN,"Entre los siglos VIII y VI a. C. los fenicios establecieron el primer sistema comercial mediterráneo, que abarcó des..."
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",NaN,Las raíces evolutivas del sufrimiento humano Una nueva comprensión de la salud mental que brinda una hipótesis revel...


In [18]:
# Guardamos el catalogo depurado para que el Paso 2 no tenga que repetir todo esto.
ruta_salida = CARPETA_SALIDAS / "catalogo_paso1.csv"
libros.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"Guardado en {ruta_salida} ({len(libros)} libros)")

Guardado en salidas\catalogo_paso1.csv (8592 libros)


**Resultado.** De los 8747 libros del archivo quedan **8592 disponibles**. Se eliminaron 155
en total: 1 sin sinopsis, 0 por identificador repetido, 98 por tener la sinopsis exactamente
igual a la de otro libro y 56 por ser la misma obra con el título escrito de otra forma.

El catálogo depurado queda en `salidas/catalogo_paso1.csv`. **Todavía incluye los libros
contaminados**: detectarlos es el trabajo del Paso 2.

# Paso 2 — Auditar el catálogo

El catálogo tiene registros contaminados de varios tipos: algunos traen enlaces evidentes,
otros los disimulan, otros solo agregan texto promocional y otros son libros inventados que
copian la sinopsis de otro. **Ningún método detecta todos**, así que construimos varios
detectores, medimos cada uno y los combinamos.

La cátedra entrega `muestra_etiquetada.csv`: 500 libros con la columna `contaminado` y, en los
contaminados, el texto insertado en la columna `fragmento`.

**Cómo la usamos.** El 70 % para *desarrollo* (ahí elegimos regex, umbrales y entrenamos) y el
30 % para *prueba* (ahí solo medimos). La división es **por libro**: si dos oraciones del mismo
libro cayeran una en cada lado, la medición saldría optimista.

Dos avisos de la consigna: la muestra está **enriquecida** al 20 % de contaminados, bastante
más que el catálogo, así que la precisión que midamos va a ser mejor que la real; y **no trae
todas las variantes**, hay mensajes en el catálogo que nuestros modelos nunca vieron.

In [19]:
import unicodedata
from collections import Counter, defaultdict

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, precision_recall_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

RUTA_MUESTRA = Path("consigna_tp3/tp3/muestra_etiquetada.csv")

# keep_default_na=False es imprescindible: si no, el "fragmento" vacio de los
# libros limpios se lee como NaN y se propaga en silencio.
muestra = pd.read_csv(RUTA_MUESTRA, keep_default_na=False)

# Normalizamos los espacios igual que en el Paso 1, para poder comparar textos.
for columna in ["synopsis", "fragmento"]:
    muestra[columna] = muestra[columna].str.replace(r"\s+", " ", regex=True).str.strip()

desarrollo, prueba = train_test_split(
    muestra, test_size=0.3, random_state=42, stratify=muestra["contaminado"]
)

print(f"desarrollo {len(desarrollo)} / prueba {len(prueba)}")
print(f"contaminados: desarrollo {desarrollo['contaminado'].sum()}, "
      f"prueba {prueba['contaminado'].sum()}")

desarrollo 350 / prueba 150
contaminados: desarrollo 70, prueba 30


In [20]:
def medir(nombre, marcado, datos):
    """Precision y recall de la clase contaminado."""
    y = datos["contaminado"]
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado, zero_division=0)
    print(f"  {nombre:34s} precision {p:.2f}   recall {r:.2f}")
    return p, r


def dividir(texto):
    """Parte un texto en oraciones y descarta los fragmentos muy cortos."""
    partes = re.split(r"(?<=[.!?…])\s+", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]


print("Oraciones de una sinopsis de ejemplo:")
for o in dividir(muestra.loc[0, "synopsis"])[:3]:
    print(f"  {o[:95]!r}")

Oraciones de una sinopsis de ejemplo:
  'La parábola, la intriga y el humor, tan escaso éste en las formas de mayor impacto comercial de'
  'El cinturón de Venus, de Harold Lawlor es significativo al respecto.'
  'Algunos de estos relatos plantean la posibilidad de una visión crítica y humorística del género'


**Resultado.** La división deja **350 libros de desarrollo y 150 de prueba**, con 70 y 30
contaminados respectivamente: la opción `stratify` mantuvo la proporción del 20 % en los dos
lados. Es el resultado que anticipa la consigna.

## 2.1 Expresiones regulares

El primer detector busca lo más evidente: enlaces, correos y teléfonos. Escribimos cuatro
patrones separados para poder medir cuánto aporta cada uno.

- `url`: direcciones con `http://`, `https://` o `www.`.
- `dominio`: el dominio **desnudo**, sin `http` ni `www`. Hace falta porque muchos mensajes
  dicen solamente "Visitá sitio.test". Pedimos al menos dos caracteres antes del punto para
  no capturar cosas como `1.es`.
- `correo`: direcciones de correo electrónico.
- `telefono`: acá hay que tener cuidado. Un patrón suelto de dígitos captura años y números
  de página de cualquier sinopsis histórica, así que exigimos el prefijo internacional `+`.

In [21]:
RE_URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)

RE_DOMINIO = re.compile(
    r"\b[a-z0-9][a-z0-9-]{1,}\.(?:com|net|org|info|biz|app|online|shop|club|xyz"
    r"|ar|es|mx|cl|co|io|me|tv|cc|invalid|example|test)\b(?:/\S*)?",
    re.IGNORECASE,
)

RE_CORREO = re.compile(r"[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}", re.IGNORECASE)

# Con el "+" obligatorio: sin eso, "publicada entre 1936 y 1939" cuenta como telefono.
RE_TELEFONO = re.compile(r"\+\d[\d\s().-]{7,}\d")

DETECTORES_REGEX = {"url": RE_URL, "dominio": RE_DOMINIO,
                    "correo": RE_CORREO, "telefono": RE_TELEFONO}


def hay_contacto(texto):
    """True si aparece un enlace, un correo o un telefono."""
    return any(patron.search(texto) for patron in DETECTORES_REGEX.values())


def que_encontro(texto):
    """Lo que encontro cada patron, para poder revisar los casos a mano."""
    return [(nombre, c) for nombre, patron in DETECTORES_REGEX.items()
            for c in patron.findall(texto)]

In [22]:
print("Cada patron por separado (desarrollo):")
for nombre, patron in DETECTORES_REGEX.items():
    medir(nombre, desarrollo["synopsis"].str.contains(patron), desarrollo)

print()
print("Los cuatro juntos:")
medir("regex (desarrollo)", desarrollo["synopsis"].map(hay_contacto), desarrollo)
medir("regex (PRUEBA)", prueba["synopsis"].map(hay_contacto), prueba)

Cada patron por separado (desarrollo):
  url                                precision 0.93   recall 0.19
  dominio                            precision 0.93   recall 0.40
  correo                             precision 1.00   recall 0.04
  telefono                           precision 1.00   recall 0.01

Los cuatro juntos:
  regex (desarrollo)                 precision 0.94   recall 0.41
  regex (PRUEBA)                     precision 1.00   recall 0.40


(1.0, 0.4)

In [23]:
print("Falsos positivos en desarrollo: sinopsis legitimas que la regex marca")
print()
falsos = desarrollo[(desarrollo["synopsis"].map(hay_contacto)) & (desarrollo["contaminado"] == 0)]
for _, libro in falsos.iterrows():
    print(f"  {libro['title']}")
    print(f"    {que_encontro(libro['synopsis'])}")

print()
print("Contaminados que la regex NO ve (los primeros seis):")
escapan = desarrollo[(~desarrollo["synopsis"].map(hay_contacto)) & (desarrollo["contaminado"] == 1)]
for f in escapan["fragmento"].head(6):
    print(f"  {f[:100]!r}")
print(f"  ... {len(escapan)} de {desarrollo['contaminado'].sum()} contaminados")

Falsos positivos en desarrollo: sinopsis legitimas que la regex marca

  Diverso.es
    [('dominio', 'Diverso.es')]
  ¿Que pasaría si…?
    [('url', 'http://www.xkcd.com'), ('dominio', 'xkcd.com')]

Contaminados que la regex NO ve (los primeros seis):
  'El libro más esperado del año llega por fin en edición digital completa. Leé este y miles de libros '
  'Conseguí este título y más de 50.000 libros en la app LetraTotal: el primer mes es sin cargo.'
  'Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.'
  'Entrá a librоexpress.invalid y leé sin límites.'
  'Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.'
  'Leé este y miles de libros más sin publicidad. Activá tu prueba gratuita desde tu celular.'
  ... 41 de 70 contaminados


**Resultado.** La regex logra **precisión 1,00 y recall 0,40 en prueba**: lo que marca está
bien marcado, pero deja pasar seis de cada diez contaminados. El patrón que más rinde es el
`dominio` desnudo (0,40 contra 0,19 de `url`): casi ningún mensaje escribe `http://`.

*¿Qué detecta bien y qué no?* Detecta la contaminación **literal**, el enlace escrito tal cual.
No ve nada de lo **ofuscado** ni del texto **promocional sin enlace**, que es el grupo más
grande.

*Los dos falsos positivos.* Un libro que se titula `Diverso.es`, y una sinopsis legítima que
menciona `http://www.xkcd.com`. Son el costo que avisa la consigna: cada libro legítimo que
eliminamos es un libro que nunca se va a poder recomendar.

## 2.2 Normalizar antes de buscar

Varios enlaces están disimulados para que la regex no los vea. En el catálogo aparecen cuatro
maniobras distintas:

| Maniobra | Ejemplo |
| --- | --- |
| Carácter invisible dentro del dominio | `capitu<U+200B>loflash.invalid` |
| Letras de otro alfabeto que se ven latinas | `libr`**о**`express.invalid` (о cirílica) |
| Letras separadas por espacios | `h t t p s : / / tomoya [.] test` |
| Signos escritos con palabras | `TomoMax punto example barra x` |

La función `normalizar(texto)` deshace las cuatro, en ese orden, y después volvemos a aplicar
la misma regex. Para ver qué carácter es el culpable usamos `unicodedata`, que da el nombre y
la categoría de cada carácter: los invisibles son de categoría `Cf`.

Dos detalles que nos costaron:

- **NFKC no sirve** para los homóglifos: una `о` cirílica no se convierte en `o` latina con
  ninguna normalización estándar, porque son letras distintas. Hay que escribir la tabla.
- El punto solo se pega si lo que sigue parece una terminación de dominio en minúscula. Si
  pegáramos todos, `"...esa historia. Es el mejor..."` daría el dominio `historia.es`.

In [24]:
# Letras de otros alfabetos que se ven igual que una latina.
HOMOGLIFOS = {
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x",
    "і": "i", "ѕ": "s", "ԁ": "d", "һ": "h", "ј": "j", "ӏ": "l", "ν": "v",
    "α": "a", "ο": "o", "ρ": "p", "ε": "e", "ι": "i", "κ": "k", "τ": "t",
}


def normalizar(texto):
    """Deja al descubierto los enlaces disimulados."""
    # 1. Caracteres invisibles (categoria Cf: zero width space, word joiner, BOM).
    texto = "".join(c for c in texto if unicodedata.category(c) != "Cf")

    # 2. Homoglifos.
    texto = "".join(HOMOGLIFOS.get(c, c) for c in texto)

    # 3. Signos escritos con palabras o entre corchetes.
    texto = re.sub(r"\s*\[\s*\.\s*\]\s*|\s*\(\s*\.\s*\)\s*", ".", texto)
    texto = re.sub(r"\s+(?:punto|dot)\s+", ".", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+(?:barra|slash)\s+", "/", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+arroba\s+", "@", texto, flags=re.IGNORECASE)

    # 4. Letras separadas por espacios: "w w w . sitio" -> "www.sitio".
    texto = re.sub(r"\b(?:[a-zA-Z0-9]\s){2,}[a-zA-Z0-9]\b",
                   lambda m: m.group(0).replace(" ", ""), texto)
    texto = re.sub(r"\s*:\s*/\s*/\s*", "://", texto)
    # Se repite porque un enlace puede tener dos puntos separados.
    for _ in range(2):
        texto = re.sub(r"(?<=\w)\s+\.\s+(?=[a-z]{2,})", ".", texto)
    texto = re.sub(r"(?<=\w)\s*/\s*(?=\w)", "/", texto)

    return texto

In [25]:
# Que caracter invisible esconde cada caso: esta es la pista que da la consigna.
ejemplo = "Visita capitu\u200bloflash.invalid para la version completa."
print("Caracteres no latinos del ejemplo:")
for c in ejemplo:
    if ord(c) > 127:
        print(f"  U+{ord(c):04X}  categoria {unicodedata.category(c)}  {unicodedata.name(c)}")

Caracteres no latinos del ejemplo:
  U+200B  categoria Cf  ZERO WIDTH SPACE


In [26]:
print("Las cuatro maniobras, antes y despues de normalizar:")
print()
for texto in ["Visita capitu\u200bloflash.invalid para la version completa.",
              "Entra a libr\u043eexpress.invalid y lee sin limites.",
              "Link en nuestro sitio: h t t p s : / / tomoya [.] test",
              "Descargalo gratis en TomoMax punto example barra x",
              "Una novela sobre la guerra. Es el mejor libro del autor."]:
    print(f"  antes   {texto!r}")
    print(f"  despues {normalizar(texto)!r}")
    print(f"  la regex encuentra: {que_encontro(normalizar(texto))}")
    print()

Las cuatro maniobras, antes y despues de normalizar:

  antes   'Visita capitu\u200bloflash.invalid para la version completa.'
  despues 'Visita capituloflash.invalid para la version completa.'
  la regex encuentra: [('dominio', 'capituloflash.invalid')]

  antes   'Entra a librоexpress.invalid y lee sin limites.'
  despues 'Entra a libroexpress.invalid y lee sin limites.'
  la regex encuentra: [('dominio', 'libroexpress.invalid')]

  antes   'Link en nuestro sitio: h t t p s : / / tomoya [.] test'
  despues 'Link en nuestro sitio: https://tomoya.test'
  la regex encuentra: [('url', 'https://tomoya.test'), ('dominio', 'tomoya.test')]

  antes   'Descargalo gratis en TomoMax punto example barra x'
  despues 'Descargalo gratis en TomoMax.example/x'
  la regex encuentra: [('dominio', 'TomoMax.example/x')]

  antes   'Una novela sobre la guerra. Es el mejor libro del autor.'
  despues 'Una novela sobre la guerra. Es el mejor libro del autor.'
  la regex encuentra: []



In [27]:
antes = desarrollo["synopsis"].map(hay_contacto)
despues = desarrollo["synopsis"].map(lambda t: hay_contacto(normalizar(t)))

print("Con el texto normalizado:")
medir("regex + normalizacion (desarrollo)", despues, desarrollo)
medir("regex + normalizacion (PRUEBA)",
      prueba["synopsis"].map(lambda t: hay_contacto(normalizar(t))), prueba)

recuperados = desarrollo[despues & ~antes]
perdidos = desarrollo[antes & ~despues]
print()
print(f"casos que la normalizacion recupero: {len(recuperados)}")
print(f"casos que la normalizacion perdio  : {len(perdidos)}")
print()
for _, libro in recuperados.head(4).iterrows():
    print(f"  contaminado={libro['contaminado']}  {libro['fragmento'][:80]!r}")
    print(f"    ahora la regex ve: {que_encontro(normalizar(libro['synopsis']))}")

Con el texto normalizado:
  regex + normalizacion (desarrollo) precision 0.95   recall 0.60


  regex + normalizacion (PRUEBA)     precision 1.00   recall 0.63

casos que la normalizacion recupero: 13
casos que la normalizacion perdio  : 0

  contaminado=1  'Entrá a librоexpress.invalid y leé sin límites.'
    ahora la regex ve: [('dominio', 'libroexpress.invalid')]
  contaminado=1  'Link en nuestro sitio: h t t p s : / / epubhub [.] example'
    ahora la regex ve: [('url', 'https://epubhub.example'), ('dominio', 'epubhub.example')]
  contaminado=1  'Link en nuestro sitio: h t t p s : / / lectorplus [.] invalid'
    ahora la regex ve: [('url', 'https://lectorplus.invalid'), ('dominio', 'lectorplus.invalid')]
  contaminado=1  'Link en nuestro sitio: h t t p s : / / tomoya [.] test'
    ahora la regex ve: [('url', 'https://tomoya.test'), ('dominio', 'tomoya.test')]


In [28]:
# Los caracteres invisibles aparecen tambien en sinopsis legitimas:
# por si solos NO son evidencia de contaminacion.
def tiene_invisible(texto):
    return any(unicodedata.category(c) == "Cf" for c in texto)

cruce = pd.crosstab(muestra["synopsis"].map(tiene_invisible), muestra["contaminado"])
cruce.index = ["sin invisibles", "con invisibles"]
print("Caracteres invisibles contra la etiqueta real, en toda la muestra:")
print(cruce.to_string())

Caracteres invisibles contra la etiqueta real, en toda la muestra:
contaminado       0   1
sin invisibles  396  91
con invisibles    4   9


**Resultado.** La normalización sube el recall en prueba de **0,40 a 0,63** sin perder
precisión. En desarrollo recuperó 13 libros y no perdió ninguno.

Los dos casos que pide la consigna se ven arriba: `capitu<U+200B>loflash.invalid`, con un ZERO
WIDTH SPACE partiendo el dominio, y `h t t p s : / / tomoya [.] test`, con las letras separadas
y el punto entre corchetes.

**La normalización no es un detector por sí misma.** La tabla muestra que hay 4 sinopsis
legítimas con caracteres invisibles, así que marcar por eso nos llevaría libros buenos. La
usamos como un **preparador** para que la regex vea lo que estaba tapado, y el motivo
`normalizacion` queda para los libros que la regex detecta **solo después** de normalizar.

## 2.3 Detectar el idioma de cada oración

Algunos mensajes insertados están en otro idioma dentro de una sinopsis en español. Primero
probamos con la **sinopsis completa** y después oración por oración, para ver por qué hace
falta bajar al nivel de la oración.

`langdetect` no es determinista salvo que se le fije la semilla, así que lo primero es
`DetectorFactory.seed = 0`. Sin eso, el notebook daría resultados distintos en cada ejecución.

In [29]:
from langdetect import detect, detect_langs, DetectorFactory, LangDetectException

DetectorFactory.seed = 0  # sin esto langdetect no es reproducible


def idioma_de(texto):
    try:
        return detect(texto)
    except LangDetectException:
        return "?"


idioma_completo = desarrollo["synopsis"].map(idioma_de)
print("Idioma de la sinopsis COMPLETA contra la etiqueta real (desarrollo):")
print(pd.crosstab(idioma_completo, desarrollo["contaminado"]).to_string())

Idioma de la sinopsis COMPLETA contra la etiqueta real (desarrollo):
contaminado    0   1
synopsis            
es           280  70


**Resultado.** Mirar la sinopsis completa **no sirve**: casi todos los contaminados siguen
dando `es`. El mensaje insertado son diez o quince palabras sobre un texto español de ciento
cincuenta, así que el idioma mayoritario no cambia. Por eso hay que bajar a la oración.

In [30]:
def idiomas_de_oraciones(texto):
    """Por cada oracion: (cantidad de palabras, idioma mas probable, probabilidad)."""
    salida = []
    for o in dividir(normalizar(texto)):
        try:
            mejor = detect_langs(o.lower())[0]
        except LangDetectException:
            continue
        salida.append((len(o.split()), mejor.lang, mejor.prob))
    return salida


print("Calculando el idioma de cada oracion (tarda un momento)...")
oraciones_desarrollo = desarrollo["synopsis"].map(idiomas_de_oraciones)
oraciones_prueba = prueba["synopsis"].map(idiomas_de_oraciones)
print("listo")

Calculando el idioma de cada oracion (tarda un momento)...


listo


Las oraciones cortas, las listas de nombres propios y las citas hacen que `langdetect` se
equivoque: una oración de cinco palabras en español puede dar `pt` o `ca`. Hay que fijar dos
parámetros, y la consigna pide justificarlos con la muestra de desarrollo:

- **un mínimo de palabras** por oración, y
- **un umbral de probabilidad** para creerle al detector.

Los elegimos con un barrido sobre desarrollo. El criterio no es el mejor F1 sino la
**precisión**: cada falso positivo es un libro legítimo que perdemos, y este detector va a
entrar en una combinación donde basta que uno marque.

In [31]:
def marca_idioma_con(lista, min_palabras, umbral):
    return any(n >= min_palabras and lang != "es" and prob >= umbral
               for n, lang, prob in lista)


filas = []
for min_palabras in [4, 6, 8, 10, 12]:
    for umbral in [0.50, 0.90, 0.99]:
        marcado = oraciones_desarrollo.map(lambda l: marca_idioma_con(l, min_palabras, umbral))
        filas.append({
            "min_palabras": min_palabras, "umbral": umbral,
            "marcados": int(marcado.sum()),
            "precision": round(precision_score(desarrollo["contaminado"], marcado, zero_division=0), 2),
            "recall": round(recall_score(desarrollo["contaminado"], marcado, zero_division=0), 2),
        })

barrido = pd.DataFrame(filas)
print("Barrido de parametros en DESARROLLO:")
print(barrido.to_string(index=False))

Barrido de parametros en DESARROLLO:
 min_palabras  umbral  marcados  precision  recall
            4    0.50        30       0.57    0.24
            4    0.90        18       0.67    0.17
            4    0.99        18       0.67    0.17
            6    0.50        20       0.75    0.21
            6    0.90        14       0.86    0.17
            6    0.99        14       0.86    0.17
            8    0.50        12       0.67    0.11
            8    0.90         8       0.88    0.10
            8    0.99         8       0.88    0.10
           10    0.50        10       0.80    0.11
           10    0.90         8       0.88    0.10
           10    0.99         8       0.88    0.10
           12    0.50         5       0.60    0.04
           12    0.90         3       0.67    0.03
           12    0.99         3       0.67    0.03


In [32]:
MIN_PALABRAS, UMBRAL_IDIOMA = 8, 0.90

d_idioma_desarrollo = oraciones_desarrollo.map(
    lambda l: marca_idioma_con(l, MIN_PALABRAS, UMBRAL_IDIOMA))
d_idioma_prueba = oraciones_prueba.map(
    lambda l: marca_idioma_con(l, MIN_PALABRAS, UMBRAL_IDIOMA))

print(f"Elegidos: minimo {MIN_PALABRAS} palabras, umbral {UMBRAL_IDIOMA}")
medir("idioma (desarrollo)", d_idioma_desarrollo, desarrollo)
medir("idioma (PRUEBA)", d_idioma_prueba, prueba)

print()
print("Oraciones que marca en desarrollo:")
for (_, libro), lista in zip(desarrollo.iterrows(), oraciones_desarrollo):
    for (n, lang, prob), o in zip(lista, dividir(normalizar(libro["synopsis"]))):
        if n >= MIN_PALABRAS and lang != "es" and prob >= UMBRAL_IDIOMA:
            marca = "contaminado" if libro["contaminado"] == 1 else "LIMPIO (falso positivo)"
            print(f"  [{marca}] detecta {lang} con {prob:.2f}")
            print(f"      {o[:100]!r}")
            break

Elegidos: minimo 8 palabras, umbral 0.9
  idioma (desarrollo)                precision 0.88   recall 0.10
  idioma (PRUEBA)                    precision 0.80   recall 0.27

Oraciones que marca en desarrollo:
  [LIMPIO (falso positivo)] detecta ca con 1.00
      'Es ésta la segunda entrega del ciclo que se inicia con Redwall.'
  [contaminado] detecta en con 1.00
      'Download this ebook for FREE, no sign-up required, at biblioya.test today.'
  [contaminado] detecta pt con 1.00
      'Baixe este livro grátis agora mesmo, sem cadastro, em librohub.test.'
  [contaminado] detecta en con 1.00
      'Download this ebook for FREE, no sign-up required, at relatoflash.test today.'
  [contaminado] detecta en con 1.00
      'Download this ebook for FREE, no sign-up required, at lectorya.test today.'
  [contaminado] detecta en con 1.00
      'Download this ebook for FREE, no sign-up required, at capitulototal.invalid today.'
  [contaminado] detecta pt con 1.00
      'IMPORTANTE: Leia milhares de 

**Resultado.** Elegimos **mínimo 8 palabras y umbral 0,90**.

El **mínimo de palabras** es el que manda: con 4 la precisión cae a 0,67 porque entran
oraciones españolas cortas que el detector confunde, y con 12 el recall se derrumba a 0,03
porque los mensajes son breves. Con 8 llega a su máximo, 0,88.

El **umbral** importa poco: entre 0,90 y 0,99 las filas son idénticas. Cuando `langdetect` se
equivoca, se equivoca con mucha confianza, así que subir la exigencia no filtra nada.

En prueba da **precisión 0,80 y recall 0,27**, y es el detector más limitado: solo unos once de
cada cien mensajes insertados están en otro idioma. Ojo que en desarrollo marca apenas 8 libros
de 350, así que con tan pocos casos las métricas se mueven mucho.

Sus falsos positivos son los que avisa la consigna: oraciones **en mayúsculas** y **listas de
nombres propios**, que `langdetect` lee como otro idioma.

## 2.4 Reconocer entidades

Aplicamos NER con spaCy (`es_core_news_lg`) a las oraciones de los fragmentos contaminados y
a una cantidad similar de oraciones limpias, para responder qué entidades aparecen en el
texto insertado y no en una sinopsis normal.

La pregunta de fondo es si el NER **sirve para detectar** o solo **para explicar** lo que
encontraron los otros detectores.

In [33]:
import spacy

# Desactivamos el lematizador: solo necesitamos las entidades.
nlp = spacy.load("es_core_news_lg", disable=["lemmatizer"])

insertadas = [o for f in desarrollo[desarrollo["contaminado"] == 1]["fragmento"]
              for o in dividir(f)]
limpias_todas = [o for s in desarrollo[desarrollo["contaminado"] == 0]["synopsis"]
                 for o in dividir(s)]
limpias = pd.Series(limpias_todas).sample(len(insertadas), random_state=42).tolist()

print(f"oraciones insertadas: {len(insertadas)}")
print(f"oraciones limpias   : {len(limpias)}")

oraciones insertadas: 88
oraciones limpias   : 88


In [34]:
def contar_entidades(oraciones):
    conteo = Counter()
    ejemplos = defaultdict(Counter)
    for doc in nlp.pipe(oraciones, batch_size=50):
        for e in doc.ents:
            conteo[e.label_] += 1
            ejemplos[e.label_][e.text] += 1
    return conteo, ejemplos


conteo_ins, ejemplos_ins = contar_entidades(insertadas)
conteo_lim, ejemplos_lim = contar_entidades(limpias)

comparacion = pd.DataFrame([
    {"etiqueta": et,
     "insertadas (cada 100 oraciones)": round(100 * conteo_ins[et] / len(insertadas), 1),
     "limpias (cada 100 oraciones)": round(100 * conteo_lim[et] / len(limpias), 1)}
    for et in sorted(set(conteo_ins) | set(conteo_lim))
])
print(comparacion.to_string(index=False))

etiqueta  insertadas (cada 100 oraciones)  limpias (cada 100 oraciones)
     LOC                             17.0                          21.6
    MISC                             42.0                          13.6
     ORG                              2.3                           5.7
     PER                             18.2                          48.9


In [35]:
for nombre, ejemplos in [("INSERTADAS", ejemplos_ins), ("LIMPIAS", ejemplos_lim)]:
    print(f"Entidades mas frecuentes en las oraciones {nombre}:")
    for et in ("MISC", "PER", "ORG", "LOC"):
        if et in ejemplos:
            top = ", ".join(f"{t} ({n})" for t, n in ejemplos[et].most_common(6))
            print(f"  {et:5s} {top}")
    print()

Entidades mas frecuentes en las oraciones INSERTADAS:
  MISC  PDF (6), Download this ebook for FREE (4), sign-up required (4), EPUB sin registro (4), Descargalo (4), EPUB (2)
  PER   compartila (7), Ganá (5), sem cadastro (1), LetraClub (1), Leia (1), sem (1)
  ORG   LetraTotal (1), com (1)
  LOC   Oferta (2), Visitá pagin⁠azona.invalid (1), Baixe (1), grátis agora mesmo (1), ventas@paginatotal.example (1), librоexpress.invalid (1)

Entidades mas frecuentes en las oraciones LIMPIAS:
  MISC  Talento (1), Hombre (1), Pasión: el futbol (1), Ahora (1), Tiempo… (1), Pueblos del Mar (1)
  PER   Cristina Roccanova (1), Alicia (1), Carlitos (1), Beremiz Samir (1), Frans de Waal (1), Boabdil (1)
  ORG   Extremoduro (1), Universidad Complutense de Madrid (1), Apple (1), Psicología Comparada (1), The New York Times «Cargado de emoción (1)
  LOC   México (2), Estado (2), Bagdad (1), Granada (1), las Alpujarras (1), Palos de la Frontera (1)



In [36]:
# Y si usaramos "la sinopsis tiene una ORG" directamente como detector?
tiene_org = [any(e.label_ == "ORG" for e in doc.ents)
             for doc in nlp.pipe(desarrollo["synopsis"].tolist(), batch_size=25)]

print("Usar 'tiene una entidad ORG' como detector:")
medir("tiene ORG (desarrollo)", pd.Series(tiene_org, index=desarrollo.index), desarrollo)
print(f"  marcaria {sum(tiene_org)} de {len(tiene_org)} libros")

Usar 'tiene una entidad ORG' como detector:
  tiene ORG (desarrollo)             precision 0.33   recall 0.41
  marcaria 89 de 350 libros


**Resultado.** Dos etiquetas se diferencian claramente:

- **`MISC` aparece tres veces más** en el texto insertado (42 por cada 100 oraciones contra 13):
  son los formatos y las marcas del spam, `PDF`, `EPUB`, `LibroLibre`, `EpubMax`.
- **`PER` es casi el doble** en las sinopsis legítimas (49 contra 18): una sinopsis real habla
  de personajes y autores, y el spam no menciona a nadie.

*¿Sirven para detectar o solo para explicar?* **Para explicar.** El detector "tiene una `ORG`"
da precisión 0,33 y marcaría 89 de los 350 libros, porque las sinopsis legítimas están llenas
de organizaciones reales (editoriales, universidades, premios).

Hay algo curioso: el NER etiqueta como `PER` o `LOC` a verbos como `Ganá`, `Visitá`,
`Descargalo`, `compartila`. Está entrenado con prosa declarativa y el spam está escrito **en
imperativo**, que las sinopsis reales no usan. Esa es la señal que el clasificador del punto
siguiente va a aprender solo.

## 2.5 Clasificar oraciones

El texto promocional sin enlaces es el grupo más grande y no hay regla que lo describa: solo
se detecta aprendiendo de ejemplos. Entrenamos dos clasificadores sobre **oraciones**, uno con
TF-IDF y otro con embeddings, los dos con la misma
`LogisticRegression(class_weight="balanced", max_iter=1000)`.

**¿Por qué clasificar oraciones y no la sinopsis completa?** Porque el mensaje insertado es
apenas un 10 % del texto: en una sinopsis de mil caracteres la señal se diluye, mientras que en
una oración el mensaje **es** el ejemplo entero. Y además así tenemos más de dos mil ejemplos
de entrenamiento en lugar de 350.

**Cómo armamos las oraciones.** Las positivas son las del `fragmento` y las negativas las del
resto de la sinopsis. Al fragmento **lo restamos antes de dividir en oraciones**, no después:
si dividiéramos primero, varias oraciones quedarían a caballo —el mensaje se insertó sin un
punto que lo separe— y ensuciarían las dos clases.

Una sinopsis es sospechosa si **alguna** de sus oraciones supera el umbral.

In [37]:
def oraciones_etiquetadas(datos):
    """Una fila por oracion. Positivas: las del fragmento insertado."""
    filas = []
    for _, libro in datos.iterrows():
        if libro["contaminado"] == 1 and libro["fragmento"]:
            sucias = dividir(libro["fragmento"])
            # Se resta el fragmento ANTES de dividir.
            resto = libro["synopsis"].replace(libro["fragmento"], " ")
        else:
            sucias, resto = [], libro["synopsis"]
        filas += [{"slug": libro["slug"], "oracion": o, "etiqueta": 1} for o in sucias]
        filas += [{"slug": libro["slug"], "oracion": o, "etiqueta": 0} for o in dividir(resto)]
    return pd.DataFrame(filas)


ora_desarrollo = oraciones_etiquetadas(desarrollo)
ora_prueba = oraciones_etiquetadas(prueba)

print(f"oraciones de desarrollo: {len(ora_desarrollo)} "
      f"(insertadas {ora_desarrollo['etiqueta'].sum()})")
print(f"oraciones de prueba    : {len(ora_prueba)} "
      f"(insertadas {ora_prueba['etiqueta'].sum()})")
print(f"desbalance: 1 insertada cada "
      f"{len(ora_desarrollo) // ora_desarrollo['etiqueta'].sum()} oraciones")
print()
print("Por eso class_weight='balanced' no es opcional: sin eso, predecir")
print("siempre 'limpia' ya acierta el 96 % de las veces.")

oraciones de desarrollo: 2310 (insertadas 88)
oraciones de prueba    : 1071 (insertadas 38)
desbalance: 1 insertada cada 26 oraciones

Por eso class_weight='balanced' no es opcional: sin eso, predecir
siempre 'limpia' ya acierta el 96 % de las veces.


In [38]:
texto_desarrollo = ora_desarrollo["oracion"].map(normalizar).tolist()
texto_prueba = ora_prueba["oracion"].map(normalizar).tolist()
y_desarrollo = ora_desarrollo["etiqueta"].values
y_prueba = ora_prueba["etiqueta"].values

# Modelo A: TF-IDF. fit_transform solo en desarrollo, transform en prueba.
vectorizador_ora = TfidfVectorizer()
A_desarrollo = vectorizador_ora.fit_transform(texto_desarrollo)
A_prueba = vectorizador_ora.transform(texto_prueba)

# Modelo B: el mismo modelo de oracion multilingue del TP2, congelado.
from sentence_transformers import SentenceTransformer
modelo_oracion = SentenceTransformer("distiluse-base-multilingual-cased-v1", device="cpu")
B_desarrollo = modelo_oracion.encode(texto_desarrollo, batch_size=64, show_progress_bar=False)
B_prueba = modelo_oracion.encode(texto_prueba, batch_size=64, show_progress_bar=False)

print(f"TF-IDF     : desarrollo {A_desarrollo.shape}  prueba {A_prueba.shape}")
print(f"embeddings : desarrollo {B_desarrollo.shape}  prueba {B_prueba.shape}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

TF-IDF     : desarrollo (2310, 11596)  prueba (1071, 11596)
embeddings : desarrollo (2310, 512)  prueba (1071, 512)


**El umbral no puede ser 0,5.** Los dos modelos no producen probabilidades comparables: con
el mismo umbral, uno marca muchos más libros que el otro. Elegimos el umbral de cada
clasificador con `precision_recall_curve` **sobre desarrollo**, buscando el primero que
alcanza una precisión de 0,90, y después los comparamos **a igual precisión**.

Un detalle de `precision_recall_curve`: devuelve un elemento más en `precision` que en
`thresholds`, así que hay que indexar con `precision[:-1]`.

In [39]:
PRECISION_OBJETIVO = 0.90
clasificadores = {}

for nombre, X_des, X_pru in [("TF-IDF", A_desarrollo, A_prueba),
                             ("embeddings", B_desarrollo, B_prueba)]:
    modelo = LogisticRegression(class_weight="balanced", max_iter=1000)
    modelo.fit(X_des, y_desarrollo)
    columna_1 = list(modelo.classes_).index(1)  # verificamos cual columna es la clase 1

    prob_des = modelo.predict_proba(X_des)[:, columna_1]
    prob_pru = modelo.predict_proba(X_pru)[:, columna_1]

    # De oracion a libro: el maximo de sus oraciones.
    libro_des = pd.Series(prob_des, index=ora_desarrollo["slug"]).groupby(level=0).max()
    libro_pru = pd.Series(prob_pru, index=ora_prueba["slug"]).groupby(level=0).max()
    etq_des = desarrollo.set_index("slug")["contaminado"].loc[libro_des.index]
    etq_pru = prueba.set_index("slug")["contaminado"].loc[libro_pru.index]

    precision, recall, umbrales = precision_recall_curve(etq_des, libro_des)
    alcanza = np.where(precision[:-1] >= PRECISION_OBJETIVO)[0]
    umbral = float(umbrales[alcanza[0]]) if len(alcanza) else 0.5

    print(f"{nombre}")
    print(f"  umbral que alcanza precision {PRECISION_OBJETIVO} en desarrollo: {umbral:.3f}")
    print(f"  con el umbral 0.5 por defecto marcaria "
          f"{int((libro_des >= 0.5).sum())} libros de {len(libro_des)}")
    medir(f"  {nombre} a nivel libro (PRUEBA)", libro_pru >= umbral,
          prueba.set_index("slug").loc[libro_pru.index])
    medir(f"  {nombre} a nivel oracion (PRUEBA)", prob_pru >= umbral,
          pd.DataFrame({"contaminado": y_prueba}))

    clasificadores[nombre] = {"modelo": modelo, "columna_1": columna_1, "umbral": umbral,
                              "libro_prueba": libro_pru >= umbral}
    print()

TF-IDF
  umbral que alcanza precision 0.9 en desarrollo: 0.262
  con el umbral 0.5 por defecto marcaria 71 libros de 350
    TF-IDF a nivel libro (PRUEBA)    precision 0.86   recall 1.00
    TF-IDF a nivel oracion (PRUEBA)  precision 0.72   recall 1.00

embeddings
  umbral que alcanza precision 0.9 en desarrollo: 0.454
  con el umbral 0.5 por defecto marcaria 74 libros de 350
    embeddings a nivel libro (PRUEBA) precision 0.86   recall 1.00
    embeddings a nivel oracion (PRUEBA) precision 0.86   recall 1.00



**Resultado, y una advertencia importante.** A nivel libro los dos modelos dan lo mismo:
precisión 0,86 y **recall 1,00** en prueba. Un recall perfecto debería dar desconfianza, y
con razón.

In [40]:
def molde(oracion):
    """La oracion con lo variable borrado: deja el molde de la frase."""
    t = normalizar(oracion).lower()
    t = re.sub(r"https?://\S+|www\.\S+|\b[\w-]+\.(?:invalid|example|test|com)\b\S*", " ENLACE ", t)
    t = re.sub(r"\+?\d[\d\s().-]{6,}\d", " TELEFONO ", t)
    t = re.sub(r"\d+", " N ", t)
    t = re.sub(r"[^a-záéíóúñü ]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


moldes_desarrollo = {molde(o) for o in ora_desarrollo[ora_desarrollo["etiqueta"] == 1]["oracion"]}
insertadas_prueba = ora_prueba[ora_prueba["etiqueta"] == 1]["oracion"]
ya_vistos = insertadas_prueba.map(lambda o: molde(o) in moldes_desarrollo)

print(f"oraciones insertadas en prueba     : {len(insertadas_prueba)}")
print(f"  con un molde que ya estaba en desarrollo: {ya_vistos.sum()}")
print(f"  con un molde NUEVO                      : {(~ya_vistos).sum()}")

oraciones insertadas en prueba     : 38
  con un molde que ya estaba en desarrollo: 38
  con un molde NUEVO                      : 0


**Resultado.** Las **38 oraciones insertadas de prueba usan un molde que ya estaba en
desarrollo**. Ninguna es nueva: los mensajes se repiten casi literales (unos 56 moldes para 100
fragmentos) y la división aleatoria los reparte entre los dos lados.

Así que ese recall 1,00 **no mide generalización, mide memorización**, y eso explica por qué
los dos modelos empatan: con la frase exacta delante, cualquiera alcanza.

Donde sí se separan es **a nivel oración**: TF-IDF da precisión 0,72 y los embeddings 0,86, con
el mismo recall. TF-IDF levanta más falsas alarmas.

La pregunta de la consigna es cuál generaliza mejor **a frases nuevas con el mismo sentido**, y
esta división no la puede responder porque no contiene ninguna. Lo probamos aparte.

In [41]:
# Frases promocionales redactadas de nuevo: mismo sentido, otro vocabulario.
promocion_nueva = [
    "Abonate al plan anual y accede a todo el fondo editorial sin restricciones.",
    "Canje tu codigo de regalo y obtene lectura ilimitada por tres meses.",
    "Invita a un conocido y los dos reciben beneficios exclusivos.",
    "Pedi el envio contra reembolso y abona recien cuando lo tengas en casa.",
    "Sumate a la comunidad de lectores y competi por premios cada semana.",
    "Reserva tu ejemplar firmado antes de que se agote el stock disponible.",
]
# Oraciones literarias inventadas, como control: no habria que marcarlas.
literatura_nueva = [
    "El relojero descubre que cada pieza del mecanismo guarda una confesion ajena.",
    "La protagonista regresa al pueblo treinta anos despues del incendio.",
    "Una investigacion sobre el origen de las mareas y su influencia en la pesca.",
    "El autor reconstruye la caida del imperio a partir de cartas privadas.",
    "Dos hermanas se reparten una herencia que ninguna de las dos queria.",
    "Un ensayo breve sobre la soledad en las ciudades industriales del norte.",
]


def probar_frases(titulo, frases, habria_que_marcar):
    print(f"{titulo}  (lo correcto: {'marcar' if habria_que_marcar else 'NO marcar'})")
    textos = [normalizar(f) for f in frases]
    representaciones = {"TF-IDF": vectorizador_ora.transform(textos),
                        "embeddings": modelo_oracion.encode(textos, show_progress_bar=False)}
    aciertos = Counter()
    for i, frase in enumerate(frases):
        linea = f"  {frase[:58]:60s}"
        for nombre, X in representaciones.items():
            c = clasificadores[nombre]
            prob = c["modelo"].predict_proba(X[i:i + 1])[0, c["columna_1"]]
            marca = prob >= c["umbral"]
            aciertos[nombre] += int(marca == habria_que_marcar)
            linea += f" | {nombre[:4]} {prob:.2f} {'SI' if marca else 'no'}"
        print(linea)
    print(f"  aciertos: TF-IDF {aciertos['TF-IDF']}/{len(frases)}, "
          f"embeddings {aciertos['embeddings']}/{len(frases)}")
    print()
    return aciertos


a = probar_frases("PROMOCION con otras palabras", promocion_nueva, True)
b = probar_frases("LITERATURA nueva (control)", literatura_nueva, False)
total = len(promocion_nueva) + len(literatura_nueva)
for nombre in ("TF-IDF", "embeddings"):
    print(f"{nombre:12s} total {a[nombre] + b[nombre]}/{total}")

PROMOCION con otras palabras  (lo correcto: marcar)
  Abonate al plan anual y accede a todo el fondo editorial s   | TF-I 0.18 no | embe 0.85 SI
  Canje tu codigo de regalo y obtene lectura ilimitada por t   | TF-I 0.59 SI | embe 0.98 SI
  Invita a un conocido y los dos reciben beneficios exclusiv   | TF-I 0.09 no | embe 0.58 SI
  Pedi el envio contra reembolso y abona recien cuando lo te   | TF-I 0.12 no | embe 0.53 SI
  Sumate a la comunidad de lectores y competi por premios ca   | TF-I 0.09 no | embe 0.86 SI
  Reserva tu ejemplar firmado antes de que se agote el stock   | TF-I 0.29 SI | embe 0.95 SI
  aciertos: TF-IDF 2/6, embeddings 6/6

LITERATURA nueva (control)  (lo correcto: NO marcar)
  El relojero descubre que cada pieza del mecanismo guarda u   | TF-I 0.09 no | embe 0.02 no
  La protagonista regresa al pueblo treinta anos despues del   | TF-I 0.10 no | embe 0.03 no
  Una investigacion sobre el origen de las mareas y su influ   | TF-I 0.03 no | embe 0.06 no
  El autor reconst

**Resultado.** Acá la diferencia es grande: con promoción redactada de otra forma, **TF-IDF
acierta 2 de 6 y los embeddings 6 de 6**. En el control literario los dos aciertan 6 de 6, así
que la ventaja no viene de marcar más.

La razón es lo que mide cada representación. TF-IDF compara **palabras**: si la frase nueva no
comparte tokens con las del entrenamiento, no la reconoce. Los embeddings comparan **sentido**:
"Abonate al plan anual" cae cerca de "Activá tu prueba gratuita" sin compartir ninguna palabra.
Por eso también reconocen el mensaje en portugués.

Al revés, para una **cadena literal** como una URL conviene TF-IDF, o mejor la regex del 2.1:
una dirección es una cadena, no un significado.

**Decisión.** En el 2.7 usamos el clasificador con **embeddings**, porque generaliza a frases
que no vio y la consigna avisa que en el catálogo hay mensajes que no están en la muestra. Los
umbrales de acá sirven para comparar las dos representaciones; para aplicarlo al catálogo lo
reajustamos en el 2.7.

## 2.6 Encontrar libros copiados

Algunos libros no existen: copian la sinopsis de otro libro del catálogo y le agregan texto.
Para encontrarlos buscamos, para cada libro, su **vecino más parecido** con la similitud del
coseno.

Usamos **TF-IDF** y no embeddings, por dos razones: la copia reproduce el texto *literal*, que
es justamente lo que TF-IDF mide bien, y el modelo de oración corta las sinopsis a 128 tokens,
así que no vería el final del texto.

Después nos quedamos con los pares de similitud alta y **autores distintos**: las series
legítimas también comparten texto, pero en general tienen el mismo autor.

In [42]:
# Los autores se leyeron del CSV como texto; los pasamos a una cadena comparable.
import ast

def autores_como_texto(valor):
    """Los autores pueden venir como lista (en memoria) o como texto (leidos de un CSV)."""
    if isinstance(valor, list):
        return ", ".join(valor)
    if isinstance(valor, str) and valor.startswith("["):
        return ", ".join(ast.literal_eval(valor))
    return str(valor)

libros["autor"] = libros["authors"].map(autores_como_texto)

vectorizador_libros = TfidfVectorizer()
M = vectorizador_libros.fit_transform(libros["synopsis"].fillna(""))
print(f"matriz TF-IDF del catalogo: {M.shape}")

# n_neighbors=2 porque el primer vecino de cada libro es el libro mismo.
vecinos = NearestNeighbors(n_neighbors=2, metric="cosine").fit(M)
distancias, indices = vecinos.kneighbors(M)
similitud = 1 - distancias[:, 1]
vecino_de = indices[:, 1]

print(f"similitud con el vecino mas parecido: media {similitud.mean():.3f}, "
      f"maxima {similitud.max():.3f}")
print(f"libros con similitud >= 0.80: {(similitud >= 0.80).sum()}")

matriz TF-IDF del catalogo: (8592, 68469)


similitud con el vecino mas parecido: media 0.250, maxima 1.000
libros con similitud >= 0.80: 241


In [43]:
def jaccard(a, b):
    """Similitud de Jaccard sobre los conjuntos de palabras."""
    A, B = set(re.findall(r"\w+", a.lower())), set(re.findall(r"\w+", b.lower()))
    return len(A & B) / len(A | B) if A | B else 0.0


SIMILITUD_MINIMA = 0.80
CARACTERES_AGREGADOS = 30   # la copia tiene texto AGREGADO, no es solo un duplicado

filas, vistos = [], set()
for i in np.where(similitud >= SIMILITUD_MINIMA)[0]:
    j = int(vecino_de[i])
    clave = tuple(sorted((int(i), j)))
    if clave in vistos:
        continue
    vistos.add(clave)
    uno, otro = libros.iloc[i], libros.iloc[j]
    largo_uno, largo_otro = len(str(uno["synopsis"])), len(str(otro["synopsis"]))
    # La copia es la mas larga: es la que trae el texto agregado.
    copia, original = (uno, otro) if largo_uno > largo_otro else (otro, uno)
    filas.append({
        "copia": copia["slug"], "copia_titulo": copia["title"], "copia_autor": copia["autor"],
        "original": original["slug"], "original_titulo": original["title"],
        "original_autor": original["autor"],
        "similitud": round(float(similitud[i]), 3),
        "jaccard": round(jaccard(str(uno["synopsis"]), str(otro["synopsis"])), 3),
        "agregados": abs(largo_uno - largo_otro),
        "mismo_autor": copia["autor"] == original["autor"],
    })

pares = pd.DataFrame(filas)
print(f"pares distintos con similitud >= {SIMILITUD_MINIMA}: {len(pares)}")
print(f"  mismo autor (serie legitima)     : {pares['mismo_autor'].sum()}")
print(f"  autores distintos                : {(~pares['mismo_autor']).sum()}")

copias = pares[(~pares["mismo_autor"]) & (pares["agregados"] >= CARACTERES_AGREGADOS)]
print(f"  y con texto agregado (son copias): {len(copias)}")

pares distintos con similitud >= 0.8: 144
  mismo autor (serie legitima)     : 78
  autores distintos                : 66
  y con texto agregado (son copias): 64


In [44]:
print("Las copias mas claras:")
print()
for _, p in copias.sort_values("similitud", ascending=False).head(5).iterrows():
    print(f"  similitud {p['similitud']:.3f}   Jaccard {p['jaccard']:.2f}   "
          f"+{p['agregados']} caracteres")
    print(f"    COPIA    {p['copia_titulo'][:46]:48s} [{p['copia_autor'][:30]}]")
    print(f"    ORIGINAL {p['original_titulo'][:46]:48s} [{p['original_autor'][:30]}]")
    print()

print("Pares con el MISMO autor, que NO marcamos (series y colecciones legitimas):")
for _, p in pares[pares["mismo_autor"]].head(4).iterrows():
    print(f"  {p['similitud']:.3f}  {p['copia_titulo'][:40]:42s} / "
          f"{p['original_titulo'][:30]:32s} [{p['copia_autor'][:22]}]")

Las copias mas claras:

  similitud 0.988   Jaccard 0.94   +159 caracteres
    COPIA    La última tormenta                               [Lucía Oyarzún]
    ORIGINAL La pequeña comunista que no sonreía nunca        [Lola Lafon]

  similitud 0.988   Jaccard 0.95   +117 caracteres
    COPIA    Los hijos de la promesa                          [Paula Arrieta]
    ORIGINAL Elvira de Hidalgo                                [Juan Villalba Sebastián]

  similitud 0.985   Jaccard 0.95   +119 caracteres
    COPIA    Bajo la isla                                     [Clara Correa]
    ORIGINAL Trás-os-Montes                                   [Julio Llamazares]

  similitud 0.984   Jaccard 0.93   +74 caracteres
    COPIA    Richard P. Feynman. La física de las palabras    [Michelle Feynman]
    ORIGINAL La física de las palabras                        [Richard Phillips Feynman]

  similitud 0.977   Jaccard 0.92   +109 caracteres
    COPIA    Crónica del bosque                               [Sofía Co

In [45]:
# Comprobacion con la muestra etiquetada: la heuristica "la copia es la mas larga"
# tendria que marcar al libro inventado y NO al original.
etiqueta_real = dict(zip(muestra["slug"], muestra["contaminado"]))

copias_en_muestra = [s for s in copias["copia"] if s in etiqueta_real]
print(f"copias que estan en la muestra etiquetada: {len(copias_en_muestra)}")
print(f"  de ellas, marcadas como contaminadas   : "
      f"{sum(etiqueta_real[s] for s in copias_en_muestra)}")

orig_en_muestra = [s for s in copias["original"] if s in etiqueta_real]
print(f"'originales' que estan en la muestra      : {len(orig_en_muestra)}")
print(f"  de ellos, marcados como contaminados   : "
      f"{sum(etiqueta_real[s] for s in orig_en_muestra)}")

copias que estan en la muestra etiquetada: 11
  de ellas, marcadas como contaminadas   : 11
'originales' que estan en la muestra      : 4
  de ellos, marcados como contaminados   : 0


**Resultado.** Con similitud ≥ 0,80 hay 144 pares. De esos, **78 tienen el mismo autor**
—series y colecciones legítimas— y **66 tienen autores distintos**. Pidiendo además que la
copia tenga 30 caracteres de más quedan **64 copias**.

La comprobación contra la muestra es la mejor noticia del punto: de las copias que aparecen
ahí, **las 11 están marcadas como contaminadas**, y de los libros que señalamos como original,
**ninguno**. La heurística "la copia es la más larga" acierta.

Todas tienen la misma firma: un autor inventado de nombre argentino y un título genérico
—*Memorias del faro*, *Los hijos de la marea*— sobre la sinopsis de un libro real.

*¿Para qué sirve el Jaccard?* Para confirmar el par: da entre 0,86 y 0,95, o sea que casi todas
las palabras de una están en la otra y lo que sobra es el texto agregado.

**Por qué pedimos los 30 caracteres.** Sin esa condición entraban dos pares que no son spam: un
libro cargado dos veces con la misma sinopsis y distinta atribución de autor, y dos números de
una antología con colaboradores distintos. Los dos tienen diferencia de longitud casi nula.

**Caso de falla.** El par *Richard P. Feynman. La física de las palabras* y *La física de las
palabras* sí entra, y es la misma obra con distinta atribución. El filtro de autores distintos
no alcanza cuando una edición figura a nombre de un compilador o un familiar.

## 2.7 Combinar y entregar

Ya tenemos cinco detectores. Los combinamos con la regla más simple: **un libro es sospechoso
si al menos uno lo marca**. Primero medimos cada detector y la combinación sobre los 150
libros de prueba, que no se usaron para elegir nada.

Los motivos usan el vocabulario del ejemplo que entrega la cátedra y se concatenan con `+`
cuando más de un detector marca el mismo libro:

| Motivo | Cuándo se asigna |
| --- | --- |
| `regex` | la regex encuentra un enlace, correo o teléfono en el texto original |
| `normalizacion` | la regex lo encuentra **solo después** de normalizar |
| `idioma` | alguna oración de 8 palabras o más está en otro idioma con probabilidad ≥ 0,90 |
| `clasificador` | alguna oración supera el umbral del clasificador con embeddings |
| `casi_duplicado` | el libro copia la sinopsis de otro y le agrega texto |

In [46]:
def probabilidad_por_libro(textos):
    """Maxima probabilidad del clasificador entre las oraciones de cada sinopsis.

    Modo produccion: se parte la sinopsis cruda, sin saber donde esta el fragmento.
    """
    pares = [(i, o) for i, t in enumerate(textos) for o in dividir(str(t))]
    if not pares:
        return pd.Series(0.0, index=range(len(textos)))
    indices = [i for i, _ in pares]
    X = modelo_oracion.encode([normalizar(o) for _, o in pares],
                              batch_size=64, show_progress_bar=False)
    c = clasificadores["embeddings"]
    prob = c["modelo"].predict_proba(X)[:, c["columna_1"]]
    return pd.Series(prob, index=indices).groupby(level=0).max().reindex(
        range(len(textos)), fill_value=0.0)


def detectar_idioma(textos):
    return pd.Series([marca_idioma_con(idiomas_de_oraciones(str(t)), MIN_PALABRAS, UMBRAL_IDIOMA)
                      for t in textos], index=range(len(textos)))

### Ajustar el umbral del clasificador antes de aplicarlo

Dos cosas cambian entre la comparación del 2.5 y el sistema que se aplica de verdad.

**En producción no sabemos dónde está el fragmento.** En 2.5 le dimos las oraciones del mensaje
ya separadas, algo que solo se puede hacer conociendo la etiqueta. Sobre el catálogo hay que
partir la sinopsis cruda, así que volvemos a medir **en desarrollo, partiéndola cruda**.

**Los dos errores no cuestan lo mismo.** Un falso negativo se recupera: si el lector encuentra
spam en el Paso 3, lo marca con motivo `revision_manual`. Un falso positivo es definitivo: ese
libro sale del catálogo y nunca se puede recomendar.

Por eso subimos el objetivo de precisión de 0,90 a **0,98**. La tabla se calcula **solo con
desarrollo**: elegir el umbral mirando prueba sería la fuga de datos que venimos evitando.

In [47]:
des = desarrollo.reset_index(drop=True)
prob_desarrollo = probabilidad_por_libro(des["synopsis"].tolist())

precision, recall, umbrales = precision_recall_curve(des["contaminado"], prob_desarrollo)
filas = []
for objetivo in [0.90, 0.95, 0.98, 1.00]:
    alcanza = np.where(precision[:-1] >= objetivo)[0]
    if not len(alcanza):
        continue
    u = float(umbrales[alcanza[0]])
    marcado = prob_desarrollo >= u
    filas.append({
        "objetivo de precision": objetivo, "umbral": round(u, 3),
        "precision (desarrollo)": round(precision_score(des["contaminado"], marcado, zero_division=0), 2),
        "recall (desarrollo)": round(recall_score(des["contaminado"], marcado, zero_division=0), 2),
    })
print("Umbral que alcanza cada objetivo, medido SOLO en desarrollo:")
print(pd.DataFrame(filas).to_string(index=False))

Umbral que alcanza cada objetivo, medido SOLO en desarrollo:
 objetivo de precision  umbral  precision (desarrollo)  recall (desarrollo)
                  0.90   0.454                    0.90                 0.93
                  0.95   0.574                    0.95                 0.89
                  0.98   0.711                    0.98                 0.87
                  1.00   0.763                    1.00                 0.84


In [48]:
PRECISION_DESEADA = 0.98

alcanza = np.where(precision[:-1] >= PRECISION_DESEADA)[0]
UMBRAL_CLASIF = float(umbrales[alcanza[0]])
print(f"Umbral elegido para aplicar el clasificador: {UMBRAL_CLASIF:.3f}")
print(f"(el de 2.5, con objetivo 0.90 y las oraciones ya separadas, era "
      f"{clasificadores['embeddings']['umbral']:.3f})")

Umbral elegido para aplicar el clasificador: 0.711
(el de 2.5, con objetivo 0.90 y las oraciones ya separadas, era 0.454)


### ¿Alcanza con que lo marque uno solo?

La regla "sospechoso si al menos uno lo marca" trata a los cinco detectores como si fueran
igual de confiables, y no lo son. Antes de aplicarla miramos, **en desarrollo**, qué pasa con
los libros que marca un único detector sin que ninguno lo acompañe.

In [49]:
d_regex = des["synopsis"].map(hay_contacto)
d_norm = des["synopsis"].map(lambda t: hay_contacto(normalizar(t)))
d_clasif = prob_desarrollo >= UMBRAL_CLASIF
d_copia = des["slug"].isin(set(copias["copia"]))
d_idioma = d_idioma_desarrollo.reset_index(drop=True)

otros = d_regex | d_norm | d_clasif | d_copia

acompanado = d_idioma & otros
solo_idioma = d_idioma & ~otros

print(f"Libros que marca el detector de idioma en desarrollo: {int(d_idioma.sum())}")
print(f"  acompanado por otro detector : {int(acompanado.sum())}  "
      f"-> contaminados {int(des.loc[acompanado, 'contaminado'].sum())}")
print(f"  SOLO el idioma               : {int(solo_idioma.sum())}  "
      f"-> contaminados {int(des.loc[solo_idioma, 'contaminado'].sum())}")
print()
for _, libro in des[solo_idioma].iterrows():
    print(f"  marcado solo por idioma: {libro['title']!r} "
          f"(contaminado={libro['contaminado']})")

Libros que marca el detector de idioma en desarrollo: 8
  acompanado por otro detector : 7  -> contaminados 7
  SOLO el idioma               : 1  -> contaminados 0

  marcado solo por idioma: 'Mossflower' (contaminado=0)


In [50]:
for nombre, comb in [("al menos uno de los cinco", otros | d_idioma),
                     ("el idioma no cuenta solo", otros)]:
    medir(f"{nombre:30s}", comb, des)
    print(f"     marca {int(comb.sum())} libros de {len(des)}")

  al menos uno de los cinco          precision 0.95   recall 0.99
     marca 73 libros de 350
  el idioma no cuenta solo           precision 0.96   recall 0.99
     marca 72 libros de 350


**Resultado.** En desarrollo el idioma marca 8 libros. **Siete** los marca también otro
detector, y esos siete están contaminados. El **único** que marca por su cuenta es legítimo.

O sea que no aporta ni un acierto propio. Tiene sentido: los mensajes en otro idioma traen
además un dominio, y de eso ya se ocupa la regex.

Sobre el catálogo el efecto es mayor: marca 139 libros y **84 sin que ningún otro lo acompañe**.
Entre esos 84 aparecen *El método Zettelkasten* o *Memorias y aventuras*, libros reales con
sinopsis normales en español donde `langdetect` se confundió con una oración en mayúsculas.

Así que cambiamos la regla: **un libro es sospechoso si lo marca la regex, la normalización, el
clasificador o el detector de copias.** El idioma se sigue midiendo y aparece en el `motivo`
cuando acompaña a otro, pero **no alcanza solo** para sacar un libro del catálogo.

In [51]:
pru = prueba.reset_index(drop=True)

p_regex = pru["synopsis"].map(hay_contacto)
p_norm = pru["synopsis"].map(lambda t: hay_contacto(normalizar(t))) & ~p_regex
p_idioma = d_idioma_prueba.reset_index(drop=True)
p_clasif = probabilidad_por_libro(pru["synopsis"].tolist()) >= UMBRAL_CLASIF
p_copia = pru["slug"].isin(set(copias["copia"]))

detectores_prueba = {
    "regex": p_regex,
    "regex + normalizacion": p_regex | p_norm,
    "idioma": p_idioma,
    "clasificador (embeddings)": p_clasif,
    "casi duplicado": p_copia,
}
# El idioma queda afuera de la decision: la celda siguiente muestra por que.
combinacion = p_regex | p_norm | p_clasif | p_copia

filas = []
etiqueta_combinacion = "COMBINACION (el idioma no cuenta solo)"
for nombre, marcado in list(detectores_prueba.items()) + [(etiqueta_combinacion, combinacion)]:
    filas.append({
        "detector": nombre,
        "libros marcados": int(marcado.sum()),
        "precision": round(precision_score(pru["contaminado"], marcado, zero_division=0), 2),
        "recall": round(recall_score(pru["contaminado"], marcado, zero_division=0), 2),
    })

tabla_final = pd.DataFrame(filas)
print("Sobre los 150 libros de PRUEBA (30 contaminados):")
print()
print(tabla_final.to_string(index=False))

Sobre los 150 libros de PRUEBA (30 contaminados):

                              detector  libros marcados  precision  recall
                                 regex               12        1.0    0.40
                 regex + normalizacion               19        1.0    0.63
                                idioma               10        0.8    0.27
             clasificador (embeddings)               25        1.0    0.83
                        casi duplicado                3        1.0    0.10
COMBINACION (el idioma no cuenta solo)               29        1.0    0.97


In [52]:
print("Falsos positivos de la combinacion: libros legitimos que perderiamos")
for _, libro in pru[combinacion & (pru["contaminado"] == 0)].iterrows():
    print(f"  {libro['title'][:60]}")

print()
print("Contaminados que se escapan:")
escapados = pru[~combinacion & (pru["contaminado"] == 1)]
for f in escapados["fragmento"]:
    print(f"  {f[:95]!r}")
if len(escapados) == 0:
    print("  ninguno")

Falsos positivos de la combinacion: libros legitimos que perderiamos

Contaminados que se escapan:
  'Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrón'


### Aplicar al catálogo completo

Ahora corremos los mismos detectores sobre los 8592 libros del Paso 1. Los dos detectores
lentos —el idioma y el clasificador, que trabajan oración por oración sobre unas sesenta mil
oraciones— guardan su resultado en `salidas/`, así que si se vuelve a ejecutar el notebook no
hay que recalcularlos.

In [53]:
import time

ruta_idioma = CARPETA_SALIDAS / "deteccion_idioma_catalogo.csv"
ruta_clasif = CARPETA_SALIDAS / "deteccion_clasificador_catalogo.csv"
sinopsis_catalogo = libros["synopsis"].fillna("").tolist()

inicio = time.time()
c_regex = libros["synopsis"].fillna("").map(hay_contacto).reset_index(drop=True)
c_norm = libros["synopsis"].fillna("").map(
    lambda t: hay_contacto(normalizar(t))).reset_index(drop=True) & ~c_regex
print(f"regex y normalizacion: {time.time() - inicio:.0f}s")

def leer_cache(ruta, columna):
    """Lee un cache guardado por posicion, comprobando que es de estos mismos libros."""
    cache = pd.read_csv(ruta)
    if list(cache["slug"]) != list(libros["slug"]):
        raise ValueError(
            f"{ruta} no corresponde a este catalogo (cambio el Paso 1). "
            "Borrar el archivo y volver a ejecutar esta celda."
        )
    return cache[columna]


if ruta_idioma.exists():
    c_idioma = leer_cache(ruta_idioma, "marcado").astype(bool)
    print("idioma: leido de salidas/")
else:
    inicio = time.time()
    c_idioma = detectar_idioma(sinopsis_catalogo)
    pd.DataFrame({"slug": libros["slug"].values, "marcado": c_idioma.values}).to_csv(
        ruta_idioma, index=False, encoding="utf-8")
    print(f"idioma: {time.time() - inicio:.0f}s")

if ruta_clasif.exists():
    prob_clasif = leer_cache(ruta_clasif, "probabilidad")
    print("clasificador: leido de salidas/")
else:
    inicio = time.time()
    prob_clasif = probabilidad_por_libro(sinopsis_catalogo)
    pd.DataFrame({"slug": libros["slug"].values, "probabilidad": prob_clasif.values}).to_csv(
        ruta_clasif, index=False, encoding="utf-8")
    print(f"clasificador: {time.time() - inicio:.0f}s")

c_clasif = prob_clasif >= UMBRAL_CLASIF
c_copia = libros["slug"].isin(set(copias["copia"])).reset_index(drop=True)

regex y normalizacion: 4s
idioma: leido de salidas/
clasificador: leido de salidas/


In [54]:
marcas = {"regex": c_regex, "normalizacion": c_norm, "idioma": c_idioma,
          "clasificador": c_clasif, "casi_duplicado": c_copia}

# El orden es fijo para que el motivo sea siempre igual ante los mismos detectores.
ORDEN_MOTIVOS = ["regex", "normalizacion", "idioma", "clasificador", "casi_duplicado"]

# El idioma no decide por si solo; los otros cuatro si.
sospechoso = c_regex | c_norm | c_clasif | c_copia

motivo = []
for i in range(len(libros)):
    if not sospechoso.iloc[i]:
        motivo.append("")
        continue
    # En un libro ya marcado, el idioma suma informacion al motivo.
    motivo.append("+".join(m for m in ORDEN_MOTIVOS if bool(marcas[m].iloc[i])))

auditado = libros.reset_index(drop=True).copy()
auditado["motivo"] = motivo

sospechosos = auditado.loc[auditado["motivo"] != "", ["slug", "motivo"]]
catalogo_limpio = auditado.loc[auditado["motivo"] == ""].drop(columns=["motivo", "autor"])

print(f"libros auditados : {len(auditado)}")
print(f"sospechosos      : {len(sospechosos)}  ({100 * len(sospechosos) / len(auditado):.1f}%)")
print(f"catalogo limpio  : {len(catalogo_limpio)}")
print()
print("Cuantos marca cada detector por su cuenta en el catalogo completo:")
for m in ORDEN_MOTIVOS:
    print(f"  {m:16s} {int(marcas[m].sum()):5d}")
print()
print(f"Libros que marca SOLO el idioma y por eso NO sacamos: "
      f"{int((c_idioma & ~sospechoso).sum())}")

libros auditados : 8592
sospechosos      : 380  (4.4%)
catalogo limpio  : 8212

Cuantos marca cada detector por su cuenta en el catalogo completo:
  regex              129
  normalizacion       64
  idioma             139
  clasificador       309
  casi_duplicado      64

Libros que marca SOLO el idioma y por eso NO sacamos: 84


In [55]:
print("Motivos mas frecuentes:")
print(sospechosos["motivo"].value_counts().head(12).to_string())

Motivos mas frecuentes:
motivo
clasificador                                 124
regex+clasificador                            49
idioma+clasificador                           39
regex                                         38
normalizacion+clasificador                    28
normalizacion                                 23
regex+clasificador+casi_duplicado             23
clasificador+casi_duplicado                   21
regex+idioma+clasificador                     13
normalizacion+clasificador+casi_duplicado     10
regex+casi_duplicado                           5
casi_duplicado                                 2


In [56]:
sospechosos.to_csv("sospechosos.csv", index=False, encoding="utf-8")
catalogo_limpio.to_csv("catalogo_limpio.csv", index=False, encoding="utf-8")

# Copias con el nombre que pide la entrega.
sospechosos.to_csv("grupo11_sospechosos.csv", index=False, encoding="utf-8")

print(f"sospechosos.csv       {len(sospechosos)} filas, columnas {list(sospechosos.columns)}")
print(f"catalogo_limpio.csv   {len(catalogo_limpio)} filas")
print(f"grupo11_sospechosos.csv (entrega)")
print()
print(sospechosos.head(8).to_string(index=False))

sospechosos.csv       380 filas, columnas ['slug', 'motivo']
catalogo_limpio.csv   8212 filas
grupo11_sospechosos.csv (entrega)

                                                                                      slug                      motivo
                                                                  un-imperio-de-ingenieros                clasificador
                                                                     el-fin-de-la-ansiedad                clasificador
                                                                       la-isla-de-robinson               normalizacion
                                                                        la-ultima-estacion clasificador+casi_duplicado
el-genero-y-nuestros-cerebros-la-nueva-neurociencia-que-rompe-el-mito-del-cerebro-femenino                       regex
                                                                   que-quedara-de-nosotros          regex+clasificador
                                      

**Resultado.** Sobre los 150 libros de prueba, que no se usaron para elegir ningún umbral ni
ninguna regla, la combinación alcanza **precisión 1,00 y recall 0,97**: detecta 29 de los 30
contaminados y no se lleva ni un libro legítimo.

Cada detector cubre un tipo distinto de contaminación, y por eso la combinación supera a todos:
la regex llega a 0,40, la normalización la sube a 0,63, el clasificador aporta 0,83 y encuentra
la promoción en prosa, y el detector de copias suma los libros inventados que ningún otro ve.

**El que se escapa** es uno solo: *"Si te gustó esta reseña, compartila con tus amigos y
participá del sorteo de un lector electrónico."* No tiene enlace, ni otro idioma, ni copia a
nadie. Es el caso más difícil, porque lo único que lo delata es el sentido de la frase.

Sobre el catálogo completo el sistema marca **380 libros (4,4 %)** y deja **8212 limpios**.

**Una advertencia sobre esa precisión de 1,00:** está medida en la muestra, que tiene un 20 % de
contaminados, y el catálogo tiene un 4 o 5 %. Con una proporción tan baja, equivocarse en el
1 % de los 8200 libros legítimos ya son 80 errores, así que la precisión real va a ser menor. La
corrección de la cátedra, con su clave oculta, es la que va a decir cuánto.

Los pasos que siguen usan **solamente** `catalogo_limpio.csv`.

# Paso 3 — Registrar las preferencias de un lector

El lector de referencia es **Franco Renna**, y responde él todas las valoraciones. La
etiqueta `interes` vale 1 si le interesaría leer el libro y 0 si no. Si no se decide, el libro
queda **sin valorar**: eso no es un 0, es una preferencia desconocida.

Recorremos el catálogo limpio en un orden fijo, mezclado una sola vez con
`sample(frac=1, random_state=42)`, hasta reunir al menos 100 respuestas válidas y 20 de cada
clase.

Las respuestas se registraron en cinco tandas de 20 libros: un asistente de IA le mostraba al
lector el título y la sinopsis de cada libro, en ese orden, y anotaba su respuesta. Las
valoraciones son todas del lector y se dieron antes de ver cualquier predicción.

In [57]:
RUTA_LIMPIO = Path("catalogo_limpio.csv")
RUTA_LOG = CARPETA_SALIDAS / "valoraciones_log.csv"
RUTA_PREFERENCIAS = Path("preferencias.csv")

catalogo_a_valorar = pd.read_csv(RUTA_LIMPIO, keep_default_na=False)
orden = catalogo_a_valorar.sample(frac=1, random_state=42).reset_index(drop=True)

# La semilla garantiza que el orden sea siempre el mismo.
otra_vez = catalogo_a_valorar.sample(frac=1, random_state=42).reset_index(drop=True)
print("el orden se repite:", orden["slug"].equals(otra_vez["slug"]))
print()
print("Los primeros cinco libros del recorrido:")
for i, libro in orden.head(5).iterrows():
    print(f"  {i}  {libro['title'][:70]}")

el orden se repite: True

Los primeros cinco libros del recorrido:
  0  Los mercenarios
  1  La fabulosa historia de Los Pelayos
  2  Número uno
  3  Segunda parte del Lazarillo de Tormes
  4  Lo que no puedo olvidar


La valoración se hace **fuera del notebook**, con el script `valorar.py`, porque lleva alrededor
de una hora y conviene poder cortar y retomar. Desde la raíz del repositorio:

```powershell
.\.venv\Scripts\python.exe TP3\valorar.py
```

Muestra un libro por pantalla y se responde con `1`, `0`, `Enter` (no me decido), `s` (es spam
que el Paso 2 no detectó) o `q` para salir. Guarda cada respuesta en el momento en
`salidas/valoraciones_log.csv`, así que al volver a abrirlo sigue donde había quedado.

In [58]:
HAY_VALORACIONES = RUTA_LOG.exists()

if HAY_VALORACIONES:
    log = pd.read_csv(RUTA_LOG, keep_default_na=False)
    log["respuesta"] = log["respuesta"].astype(str)
    print(f"respuestas registradas: {len(log)}")
    print(log["respuesta"].value_counts().rename({"1": "me interesa (1)", "0": "no (0)",
                                                  "": "no me decido", "s": "spam"}).to_string())
else:
    log = pd.DataFrame(columns=["orden", "slug", "respuesta", "momento"])
    print("Todavia no hay valoraciones.")
    print("Hay que correr 'python TP3/valorar.py' antes de seguir con los pasos 4 a 8.")

respuestas registradas: 100
respuesta
no (0)             70
me interesa (1)    30


Con el registro armamos `preferencias.csv`. Las respuestas `1` y `0` van a la columna `interes`;
las abstenciones y los libros marcados como spam quedan con `interes` **vacío** y se conservan
en el archivo, como pide la consigna.

In [59]:
if HAY_VALORACIONES:
    # De la respuesta del lector a la etiqueta: "s" y "" quedan sin valorar.
    a_interes = {"1": 1, "0": 0}
    valoraciones = log.assign(interes=log["respuesta"].map(a_interes))

    preferencias = (orden.merge(valoraciones[["slug", "respuesta", "interes"]], on="slug")
                         .loc[:, ["slug", "title", "synopsis", "interes", "respuesta"]])

    validas = preferencias["interes"].notna()
    unos = int((preferencias["interes"] == 1).sum())
    ceros = int((preferencias["interes"] == 0).sum())

    print(f"libros vistos por el lector : {len(preferencias)}")
    print(f"  interes = 1               : {unos}")
    print(f"  interes = 0               : {ceros}")
    print(f"  sin valorar               : {int((~validas).sum())}")
    print()
    print(f"objetivo de la consigna (100 validas, 20 de cada clase): "
          f"{'cumplido' if unos + ceros >= 100 and unos >= 20 and ceros >= 20 else 'todavia no'}")

    preferencias.drop(columns=["respuesta"]).to_csv(RUTA_PREFERENCIAS, index=False,
                                                    encoding="utf-8")
    print(f"\nGuardado {RUTA_PREFERENCIAS}")
else:
    preferencias = pd.DataFrame(columns=["slug", "title", "synopsis", "interes", "respuesta"])
    print("Sin valoraciones todavia: no se genera preferencias.csv.")

libros vistos por el lector : 100
  interes = 1               : 30
  interes = 0               : 70
  sin valorar               : 0

objetivo de la consigna (100 validas, 20 de cada clase): cumplido

Guardado preferencias.csv


Si el lector encontró un libro contaminado que el Paso 2 no detectó, lo agregamos a
`sospechosos.csv` con el motivo `revision_manual`. Se reconstruye el archivo entero a partir
del resultado del Paso 2 más estos, para que volver a ejecutar el notebook no duplique filas.

In [60]:
if HAY_VALORACIONES:
    marcados_a_mano = preferencias.loc[preferencias["respuesta"] == "s", "slug"]
else:
    marcados_a_mano = pd.Series(dtype=str)

manuales = pd.DataFrame({"slug": marcados_a_mano, "motivo": "revision_manual"})
# sospechosos viene del Paso 2; si un slug ya estaba, se le suma el motivo.
sospechosos_final = pd.concat([sospechosos, manuales], ignore_index=True)
sospechosos_final = (sospechosos_final.groupby("slug", sort=False)["motivo"]
                     .apply(lambda m: "+".join(dict.fromkeys("+".join(m).split("+"))))
                     .reset_index())

sospechosos_final.to_csv("sospechosos.csv", index=False, encoding="utf-8")
sospechosos_final.to_csv("grupo11_sospechosos.csv", index=False, encoding="utf-8")

print(f"sospechosos del Paso 2       : {len(sospechosos)}")
print(f"agregados por revision manual: {len(manuales)}")
print(f"total en sospechosos.csv     : {len(sospechosos_final)}")

sospechosos del Paso 2       : 380
agregados por revision manual: 0
total en sospechosos.csv     : 380


**Resultado.** Queda `preferencias.csv` con una fila por libro visto: `slug`, `title`,
`synopsis` e `interes`. Los pasos 4 a 8 trabajan solo con las filas que tienen `interes` igual
a 0 o 1; las abstenciones se conservan en el archivo pero no entran al modelo.

# Paso 4 — Separar los libros para aprender y evaluar

Solo entran los libros con `interes` igual a 0 o 1. El 80 % se usa para entrenar y el 20 % para
probar, con `stratify` para que los dos conjuntos tengan una proporción parecida de libros que le
interesan al lector. Esta división se hace **una sola vez** y la usan todos los modelos.

In [61]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix, f1_score)
from sklearn.metrics.pairwise import cosine_similarity

# keep_default_na=False: un interes vacio (abstencion) queda como "" y no se convierte en NaN ni en 0
preferencias = pd.read_csv(RUTA_PREFERENCIAS, keep_default_na=False)
# Solo 0 y 1: un vacio no se convierte en nada, queda afuera (to_numeric lo vuelve NaN y isin lo descarta)
interes_numerico = pd.to_numeric(preferencias["interes"], errors="coerce")
valoradas = preferencias[interes_numerico.isin([0, 1])].copy()
valoradas["interes"] = interes_numerico[valoradas.index].astype(int)

entrenamiento, prueba_pref = train_test_split(
    valoradas, test_size=0.2, random_state=42, stratify=valoradas["interes"]
)
print(len(entrenamiento), len(prueba_pref))

80 20


In [62]:
# Columna conjunto: entrenamiento o prueba. Las abstenciones (si las hubiera) quedan con conjunto vacio.
preferencias["conjunto"] = ""
preferencias.loc[preferencias["slug"].isin(entrenamiento["slug"]), "conjunto"] = "entrenamiento"
preferencias.loc[preferencias["slug"].isin(prueba_pref["slug"]), "conjunto"] = "prueba"

preferencias.to_csv(RUTA_PREFERENCIAS, index=False, encoding="utf-8")
preferencias.to_csv("grupo11_preferencias.csv", index=False, encoding="utf-8")   # nombre de la entrega

resumen_division = pd.DataFrame({
    "libros": [len(entrenamiento), len(prueba_pref)],
    "interes = 1": [int(entrenamiento["interes"].sum()), int(prueba_pref["interes"].sum())],
    "interes = 0": [int((entrenamiento["interes"] == 0).sum()), int((prueba_pref["interes"] == 0).sum())],
}, index=["entrenamiento", "prueba"])
resumen_division["proporcion de 1"] = (resumen_division["interes = 1"] / resumen_division["libros"]).round(2)
print(resumen_division.to_string())
print()
print("Ningun libro esta en los dos conjuntos:", set(entrenamiento["slug"]).isdisjoint(prueba_pref["slug"]))
print(f"preferencias.csv y grupo11_preferencias.csv: {len(preferencias)} filas, columnas {list(preferencias.columns)}")

               libros  interes = 1  interes = 0  proporcion de 1
entrenamiento      80           24           56              0.3
prueba             20            6           14              0.3

Ningun libro esta en los dos conjuntos: True
preferencias.csv y grupo11_preferencias.csv: 100 filas, columnas ['slug', 'title', 'synopsis', 'interes', 'conjunto']


**Resultado.** De los 100 libros valorados quedan **80 para entrenar y 20 para probar**. La
proporción de libros que le interesan al lector es la misma en los dos conjuntos: 24 de 80 en
entrenamiento y 6 de 20 en prueba (0,30). No hubo abstenciones, así que la columna `conjunto`
está completa. La división queda guardada en `preferencias.csv` y en `grupo11_preferencias.csv`.

# Paso 5 — Entrenar dos clasificadores

Los dos modelos reciben la sinopsis y aprenden `interes`, con la misma regresión logística fija:
`C=1`, `max_iter=1000` y `class_weight="balanced"`. No hay búsqueda de hiperparámetros.

`class_weight="balanced"` da más peso a los errores en la clase menos frecuente. Con 24 libros que le
interesan contra 56 que no, sin ese ajuste la regresión tiende a predecir 0 para todo, porque acertar
la clase mayoritaria es lo que más reduce el error.

In [63]:
def regresion():
    # La configuracion fija que pide la consigna, igual para los dos modelos
    return LogisticRegression(C=1, max_iter=1000, class_weight="balanced")

y_entrenamiento = entrenamiento["interes"].values
y_prueba_pref = prueba_pref["interes"].values

# Modelo A: TF-IDF por defecto. El vocabulario y los pesos se aprenden SOLO con las 80 sinopsis de entrenamiento.
tfidf_pref = TfidfVectorizer()
X_A_entrenamiento = tfidf_pref.fit_transform(entrenamiento["synopsis"])
X_A_prueba = tfidf_pref.transform(prueba_pref["synopsis"])

modelo_A = regresion().fit(X_A_entrenamiento, y_entrenamiento)
prediccion_A = modelo_A.predict(X_A_prueba)

print(f"Modelo A (TF-IDF)     entrenamiento {X_A_entrenamiento.shape}   prueba {X_A_prueba.shape}")
print(f"  predicciones en prueba: {prediccion_A.tolist()}")

Modelo A (TF-IDF)     entrenamiento (80, 4118)   prueba (20, 4118)
  predicciones en prueba: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]


In [64]:
# Modelo B: el mismo modelo de oracion multilingue del TP2 (cargado en el Paso 2), congelado.
# Recibe el texto natural; lo unico que se entrena es la regresion logistica.
X_B_entrenamiento = modelo_oracion.encode(entrenamiento["synopsis"].tolist(), normalize_embeddings=True,
                                          batch_size=32, show_progress_bar=False)
X_B_prueba = modelo_oracion.encode(prueba_pref["synopsis"].tolist(), normalize_embeddings=True,
                                   batch_size=32, show_progress_bar=False)

modelo_B = regresion().fit(X_B_entrenamiento, y_entrenamiento)
prediccion_B = modelo_B.predict(X_B_prueba)

print(f"Modelo B (embeddings) entrenamiento {X_B_entrenamiento.shape}   prueba {X_B_prueba.shape}")
print(f"  predicciones en prueba: {prediccion_B.tolist()}")

Modelo B (embeddings) entrenamiento (80, 512)   prueba (20, 512)
  predicciones en prueba: [0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1]


**Resultado.** La matriz TF-IDF tiene **4118 columnas**, una por palabra del vocabulario de las 80
sinopsis de entrenamiento, y casi todas sus celdas valen cero. La de embeddings tiene **512
columnas** densas, las del modelo de oración. En los dos casos la prueba se representa con lo que
se aprendió en entrenamiento: el mismo vocabulario y el mismo modelo.

Con la misma regresión logística, los dos modelos se comportan muy distinto. **TF-IDF predice un
solo 1** entre los 20 libros de prueba; los embeddings predicen ocho. La razón aparece en el Paso 6:
las probabilidades de TF-IDF quedan todas entre 0,40 y 0,50.

# Paso 6 — Comprobar las predicciones

Se comparan las predicciones con las respuestas reales de los 20 libros de prueba. Para la clase 1:

| Medida | Pregunta que responde |
|---|---|
| Precisión | De los libros que predijimos de interés, ¿cuántos le interesaban al lector? |
| Recall | De los libros que le interesaban, ¿cuántos detectamos? |
| F1 | ¿Qué balance logramos entre las dos? |

In [65]:
for nombre, prediccion in [("Modelo A (TF-IDF)", prediccion_A), ("Modelo B (embeddings)", prediccion_B)]:
    print(f"=== {nombre} ===")
    # zero_division=0: si un modelo no predijera ningun 1, la precision vale 0 en lugar de dar error
    print(classification_report(y_prueba_pref, prediccion, digits=2, zero_division=0))
    matriz = confusion_matrix(y_prueba_pref, prediccion, labels=[0, 1])
    print(pd.DataFrame(matriz, index=["real 0", "real 1"], columns=["predicho 0", "predicho 1"]).to_string())
    print()

=== Modelo A (TF-IDF) ===
              precision    recall  f1-score   support

           0       0.74      1.00      0.85        14
           1       1.00      0.17      0.29         6

    accuracy                           0.75        20
   macro avg       0.87      0.58      0.57        20
weighted avg       0.82      0.75      0.68        20

        predicho 0  predicho 1
real 0          14           0
real 1           5           1

=== Modelo B (embeddings) ===
              precision    recall  f1-score   support

           0       0.75      0.64      0.69        14
           1       0.38      0.50      0.43         6

    accuracy                           0.60        20
   macro avg       0.56      0.57      0.56        20
weighted avg       0.64      0.60      0.61        20

        predicho 0  predicho 1
real 0           9           5
real 1           3           3



In [66]:
def metricas_clase_1(prediccion):
    # Precision, recall y F1 de la clase 1, mas los conteos que las explican
    vp = int(((prediccion == 1) & (y_prueba_pref == 1)).sum())
    fp = int(((prediccion == 1) & (y_prueba_pref == 0)).sum())
    fn = int(((prediccion == 0) & (y_prueba_pref == 1)).sum())
    precision_1 = vp / (vp + fp) if vp + fp else 0.0
    recall_1 = vp / (vp + fn) if vp + fn else 0.0
    return {"precision": round(precision_1, 2), "recall": round(recall_1, 2),
            "F1": round(f1_score(y_prueba_pref, prediccion, zero_division=0), 2),
            "aciertos": int((prediccion == y_prueba_pref).sum()), "errores": int((prediccion != y_prueba_pref).sum()),
            "positivos predichos": int(prediccion.sum()), "verdaderos positivos": vp}

tabla_clase_1 = pd.DataFrame({"Modelo A (TF-IDF)": metricas_clase_1(prediccion_A),
                              "Modelo B (embeddings)": metricas_clase_1(prediccion_B)}).T
print("Clase 1 (le interesa) sobre los 20 libros de prueba:")
print(tabla_clase_1.to_string())

Clase 1 (le interesa) sobre los 20 libros de prueba:
                       precision  recall    F1  aciertos  errores  positivos predichos  verdaderos positivos
Modelo A (TF-IDF)           1.00    0.17  0.29      15.0      5.0                  1.0                   1.0
Modelo B (embeddings)       0.38    0.50  0.43      12.0      8.0                  8.0                   3.0


### Validación cruzada

Con 20 libros de prueba, un solo libro mueve la precisión o el recall en 0,15 o más. Para ver cuánto
varían las métricas, se repite el entrenamiento sobre los **80 libros de entrenamiento** con
`RepeatedStratifiedKFold` (5 particiones, 10 repeticiones). En el modelo A el `TfidfVectorizer` va
dentro de un `Pipeline`, así cada partición aprende su vocabulario solo con su parte de entrenamiento.
La validación cruzada **no se usa para elegir parámetros**: la configuración es la misma de antes.

In [67]:
validacion = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

pipeline_A = Pipeline([("tfidf", TfidfVectorizer()), ("regresion", regresion())])
f1_A = cross_val_score(pipeline_A, entrenamiento["synopsis"], y_entrenamiento, cv=validacion, scoring="f1")
# Los embeddings no aprenden nada de los datos (el modelo esta congelado), asi que se pueden calcular una vez
f1_B = cross_val_score(regresion(), X_B_entrenamiento, y_entrenamiento, cv=validacion, scoring="f1")

tabla_cv = pd.DataFrame({
    "F1 clase 1 (media)": [f1_A.mean(), f1_B.mean()],
    "desvio": [f1_A.std(), f1_B.std()],
    "minimo": [f1_A.min(), f1_B.min()],
    "maximo": [f1_A.max(), f1_B.max()],
}, index=["Modelo A (TF-IDF)", "Modelo B (embeddings)"]).round(2)
print(f"Validacion cruzada sobre entrenamiento: {len(f1_A)} particiones por modelo")
print(tabla_cv.to_string())

Validacion cruzada sobre entrenamiento: 50 particiones por modelo
                       F1 clase 1 (media)  desvio  minimo  maximo
Modelo A (TF-IDF)                    0.01    0.07     0.0    0.33
Modelo B (embeddings)                0.48    0.17     0.0    0.75


### Dos libros de prueba, uno por uno

In [68]:
# La probabilidad de interes=1: se busca la columna de la clase 1 en classes_ en lugar de suponerla
assert list(modelo_A.classes_) == [0, 1] and list(modelo_B.classes_) == [0, 1]
probabilidad_A = modelo_A.predict_proba(X_A_prueba)[:, list(modelo_A.classes_).index(1)]
probabilidad_B = modelo_B.predict_proba(X_B_prueba)[:, list(modelo_B.classes_).index(1)]

detalle_prueba = prueba_pref[["slug", "title", "synopsis", "interes"]].reset_index(drop=True).assign(
    pred_A=prediccion_A, prob_A=probabilidad_A.round(2), pred_B=prediccion_B, prob_B=probabilidad_B.round(2))

def mostrar_libro(fila, motivo):
    # Titulo, comienzo de la sinopsis, respuesta real y lo que dijo cada modelo
    print(f"{motivo}: {fila['title']}")
    print(f"  {fila['synopsis'][:330]}...")
    print(f"  valoracion real: {fila['interes']}  |  TF-IDF predice {fila['pred_A']} ({fila['prob_A']:.2f})"
          f"  |  embeddings predice {fila['pred_B']} ({fila['prob_B']:.2f})")
    print()

# Un acierto del modelo B en la clase 1, y su error mas seguro (el de mayor o menor probabilidad equivocada)
aciertos_1 = detalle_prueba[(detalle_prueba["interes"] == 1) & (detalle_prueba["pred_B"] == 1)]
errores_B = detalle_prueba[detalle_prueba["interes"] != detalle_prueba["pred_B"]]
if len(aciertos_1):
    mostrar_libro(aciertos_1.sort_values("prob_B", ascending=False).iloc[0], "ACIERTO (verdadero positivo)")
if len(errores_B):
    error = errores_B.assign(seguridad=(errores_B["prob_B"] - 0.5).abs()).sort_values("seguridad", ascending=False).iloc[0]
    tipo = "FALSO POSITIVO" if error["pred_B"] == 1 else "FALSO NEGATIVO"
    mostrar_libro(error, f"ERROR ({tipo})")

print("Los 20 libros de prueba:")
print(detalle_prueba.drop(columns=["synopsis", "slug"]).to_string())

ACIERTO (verdadero positivo): Lo que nos queda del mundo
  Cuando Andrew se topa por casualidad con la casa de Jamie, está herido, muerto de hambre y ya no tiene nada que perder. Un patógeno letal ha acabado con gran parte de la población mundial, incluidos todos los seres queridos de los dos chicos. Si algo han aprendido en este nuevo mundo es que deben tener miedo de lo que otras pers...
  valoracion real: 1  |  TF-IDF predice 0 (0.50)  |  embeddings predice 1 (0.70)

ERROR (FALSO POSITIVO): Variación
  Para sanar, necesitas sumergirte en las profundidades . Allie Rousseau es bailarina de ballet profesional y sabe lo que es la presión: su madre nunca le ha quitado un ojo de encima y espera la perfección cueste lo que cueste. Pero cuando una lesión pone en peligro todo por lo que Allie se ha sacrificado, decide regresar a su cas...
  valoracion real: 0  |  TF-IDF predice 0 (0.49)  |  embeddings predice 1 (0.58)

Los 20 libros de prueba:
                                                

**Resultado.**

| | Precisión (clase 1) | Recall (clase 1) | F1 (clase 1) | Aciertos / 20 | F1 en validación cruzada |
|---|---|---|---|---|---|
| A — TF-IDF | 1,00 (1 de 1) | 0,17 (1 de 6) | 0,29 | 15 | 0,01 ± 0,07 |
| B — embeddings | 0,38 (3 de 8) | 0,50 (3 de 6) | 0,43 | 12 | 0,48 ± 0,17 |

**TF-IDF casi no aprende.** Con 80 sinopsis y 4118 palabras, cada libro de prueba comparte muy
pocas palabras con los de entrenamiento, y la regresión termina dando a todos una probabilidad
parecida, entre 0,40 y 0,50. Por eso predice 0 casi siempre. Su precisión de 1,00 es un solo
acierto, y sus 15 aciertos sobre 20 vienen de predecir la clase mayoritaria: es lo que se obtiene
diciendo "no" a todo (14 de 20). La validación cruzada lo confirma: en casi todas las 50
particiones no predice ningún positivo, y su F1 medio es 0,01.

**Los embeddings sí separan algo.** Encuentran la mitad de los libros que le interesan al lector,
a costa de cinco falsos positivos. En validación cruzada su F1 va de 0 a 0,75 según la partición,
con media 0,48: el valor de prueba (0,43) está dentro de ese rango. Con tan pocos libros, el desvío
de 0,17 es parte del resultado; un solo libro cambia el F1 de prueba en más de 0,1.

**Los dos libros de arriba.**

- *Lo que nos queda del mundo* (acierto): una novela juvenil sobre dos chicos que sobreviven a una
  pandemia. El lector la valoró con 1, y los embeddings la ubican con 0,70, la probabilidad más alta
  de la prueba. Entre los libros que le interesan en entrenamiento hay ciencia ficción y fantasía
  con mundos amenazados. TF-IDF le da 0,50, justo por debajo del umbral, y la deja afuera.
- *Variación* (falso positivo): una novela romántica juvenil sobre una bailarina. Los embeddings le
  dan 0,58 y el lector la valoró con 0. Una hipótesis: el modelo reconoce el registro de novela
  juvenil, que comparte con libros que sí le interesan, pero no el tema, que al lector no le
  atrae. El modelo aprende de 24 ejemplos positivos y no tiene cómo distinguir con precisión entre
  "novela juvenil de aventuras" y "novela juvenil romántica".

# Paso 7 — Convertir las predicciones en recomendaciones

Para recomendar hace falta ordenar. Cada clasificador da la probabilidad de `interes=1` y se toman
los cinco libros de prueba con mayor probabilidad. La tercera lista no usa clasificador: el **perfil
del lector** es el promedio de los embeddings de los libros que le interesan **de entrenamiento**, y
se ordenan los libros de prueba por su similitud del coseno con ese perfil.

Las tres listas salen de **los mismos 20 libros de prueba**. `precision@5` es cuántos de los cinco
primeros tenían `interes=1`, dividido 5.

In [69]:
# Perfil del lector: SOLO los positivos de entrenamiento (los de prueba no se pueden mirar)
perfil = X_B_entrenamiento[y_entrenamiento == 1].mean(axis=0, keepdims=True)
similitud_perfil = cosine_similarity(perfil, X_B_prueba)[0]
print(f"perfil construido con {int((y_entrenamiento == 1).sum())} libros de entrenamiento con interes=1")

puntuaciones = {"TF-IDF + clasificador": probabilidad_A,
                "embeddings + clasificador": probabilidad_B,
                "similitud con el perfil": similitud_perfil}

listas, filas_p5 = {}, []
for nombre, puntaje in puntuaciones.items():
    orden_top = np.argsort(-puntaje)[:5]
    top = detalle_prueba.iloc[orden_top][["title", "interes"]].assign(puntuacion=puntaje[orden_top].round(3))
    listas[nombre] = top
    filas_p5.append({"lista": nombre, "aciertos en el top 5": int(top["interes"].sum()),
                     "precision@5": top["interes"].sum() / 5})

azar = y_prueba_pref.mean()
tabla_p5 = pd.DataFrame(filas_p5)
print()
print(tabla_p5.to_string(index=False))
print(f"\nreferencia, elegir al azar: {int(y_prueba_pref.sum())} de {len(y_prueba_pref)} libros de prueba tienen interes=1 -> {azar:.2f}")
print(f"maximo posible: con {int(y_prueba_pref.sum())} positivos en prueba, precision@5 puede llegar a {min(int(y_prueba_pref.sum()), 5) / 5:.2f}")

perfil construido con 24 libros de entrenamiento con interes=1

                    lista  aciertos en el top 5  precision@5
    TF-IDF + clasificador                     2          0.4
embeddings + clasificador                     3          0.6
  similitud con el perfil                     2          0.4

referencia, elegir al azar: 6 de 20 libros de prueba tienen interes=1 -> 0.30
maximo posible: con 6 positivos en prueba, precision@5 puede llegar a 1.00


In [70]:
for nombre, top in listas.items():
    print(f"=== {nombre} ===")
    print(top.rename(columns={"title": "titulo", "interes": "valoracion real"}).to_string(index=False))
    print()

=== TF-IDF + clasificador ===
                     titulo  valoracion real  puntuacion
      La habitación sellada                1       0.504
                 Número uno                0       0.499
 Lo que nos queda del mundo                1       0.499
Entre el brocal y la fragua                0       0.491
                  Variación                0       0.490

=== embeddings + clasificador ===
                    titulo  valoracion real  puntuacion
Lo que nos queda del mundo                1       0.702
     La habitación sellada                1       0.626
                 Variación                0       0.579
                    Ramata                0       0.535
 El jardín de los hechizos                1       0.533

=== similitud con el perfil ===
                               titulo  valoracion real  puntuacion
                             El Hakim                0       0.494
           Lo que nos queda del mundo                1       0.475
      Los funerales de 

**Resultado.**

| Lista | Aciertos en el top 5 | precision@5 |
|---|---|---|
| TF-IDF + clasificador | 2 | 0,40 |
| embeddings + clasificador | 3 | 0,60 |
| similitud con el perfil | 2 | 0,40 |
| *azar (proporción de 1 en prueba)* | — | *0,30* |

Las tres listas superan al azar, y **el clasificador con embeddings recomienda mejor que la
similitud con el perfil**: acierta 3 de 5 contra 2 de 5. La diferencia es de un solo libro sobre
20, así que por sí sola no es concluyente; lo que la respalda es la validación cruzada del Paso 6,
donde el modelo B fue consistentemente mejor que TF-IDF.

Una explicación de por qué el clasificador supera al perfil: el perfil es el promedio de los 24
libros que le interesan, que mezclan ciencia ficción, fantasía, suspenso y divulgación. Ese
promedio queda en un punto intermedio que se parece un poco a todo; de hecho, sus tres primeros
lugares tienen similitudes casi iguales (0,49 a 0,47) y son libros que el lector rechazó, como
*El Hakim* o *Segunda parte del Lazarillo de Tormes*. El clasificador, en cambio, aprende también de
los 56 libros que **no** le interesan, y eso le permite alejarse de ellos.

El top 5 de TF-IDF hay que leerlo con cuidado: sus puntuaciones van de 0,490 a 0,504, así que el
orden se decide por diferencias mínimas y sus dos aciertos dependen bastante del azar.

# Paso 8 — Recomendar libros del catálogo completo

Ahora el clasificador con embeddings del Paso 5 se aplica a los libros del catálogo limpio que el
lector **no valoró**, sin volver a entrenarlo. Quedan afuera todos los libros de `preferencias.csv`
(incluidas las abstenciones, si las hubiera) y además cualquier otra versión de un libro ya valorado,
con el mismo criterio de casi duplicados del Paso 1: mismo autor, título parecido y sin números de
volumen distintos.

In [71]:
def misma_obra(titulo_1, sinopsis_1, titulo_2, sinopsis_2):
    # El criterio del Paso 1, para dos libros del mismo autor
    parecido = fuzz.token_sort_ratio(titulo_base(titulo_1), titulo_base(titulo_2))
    if parecido < TITULO_PARECIDO or marcas_de_volumen(titulo_1) != marcas_de_volumen(titulo_2):
        return False
    return parecido >= TITULO_IGUAL or fuzz.token_sort_ratio(sinopsis_1, sinopsis_2) > SINOPSIS_PARECIDA

catalogo_rec = catalogo_a_valorar.copy()
catalogo_rec["autor"] = catalogo_rec["authors"].map(autores_como_texto)

vistos = set(preferencias["slug"])                     # valorados y abstenciones
candidatos = catalogo_rec[~catalogo_rec["slug"].isin(vistos)].copy()

# Otras versiones de los libros vistos: se comparan solo contra libros del mismo autor
otras_versiones = []
vistos_con_autor = catalogo_rec[catalogo_rec["slug"].isin(vistos)]
for _, libro in vistos_con_autor.iterrows():
    mismo_autor = candidatos[candidatos["autor"] == libro["autor"]]
    for _, cand in mismo_autor.iterrows():
        if misma_obra(libro["title"], libro["synopsis"], cand["title"], cand["synopsis"]):
            otras_versiones.append((cand["slug"], cand["title"], libro["title"]))
candidatos = candidatos[~candidatos["slug"].isin({s for s, _, _ in otras_versiones})]

print(f"catalogo limpio            : {len(catalogo_rec)}")
print(f"vistos por el lector       : {len(vistos)}")
print(f"otras versiones de vistos  : {len(otras_versiones)}")
for _, cand, original in otras_versiones:
    print(f"   {cand!r}  (version de {original!r})")
print(f"candidatos a recomendar    : {len(candidatos)}")

catalogo limpio            : 8212
vistos por el lector       : 100
otras versiones de vistos  : 0
candidatos a recomendar    : 8112


In [72]:
# Embeddings de los candidatos. Se guardan en salidas/ para no recalcularlos en cada ejecucion.
ruta_emb = CARPETA_SALIDAS / "embeddings_candidatos.npy"
ruta_emb_slugs = CARPETA_SALIDAS / "embeddings_candidatos_slugs.csv"
if ruta_emb.exists() and ruta_emb_slugs.exists() and \
        pd.read_csv(ruta_emb_slugs, keep_default_na=False)["slug"].tolist() == candidatos["slug"].tolist():
    X_candidatos = np.load(ruta_emb)
    print("embeddings de los candidatos: leidos de salidas/")
else:
    inicio = time.time()
    X_candidatos = modelo_oracion.encode(candidatos["synopsis"].tolist(), normalize_embeddings=True,
                                         batch_size=64, show_progress_bar=False)
    np.save(ruta_emb, X_candidatos)
    candidatos[["slug"]].to_csv(ruta_emb_slugs, index=False, encoding="utf-8")
    print(f"embeddings de los candidatos: {time.time() - inicio:.0f}s")

# El MISMO modelo B del Paso 5, sin volver a ajustarlo: solo predict_proba
candidatos["puntuacion"] = modelo_B.predict_proba(X_candidatos)[:, list(modelo_B.classes_).index(1)]
print(f"matriz de candidatos: {X_candidatos.shape}")
print(f"probabilidad de interes=1: media {candidatos['puntuacion'].mean():.2f}, "
      f"libros con probabilidad >= 0.5: {int((candidatos['puntuacion'] >= 0.5).sum())}")

embeddings de los candidatos: 258s
matriz de candidatos: (8112, 512)
probabilidad de interes=1: media 0.46, libros con probabilidad >= 0.5: 2654


In [73]:
top_catalogo = candidatos.sort_values("puntuacion", ascending=False).head(5)
for posicion, (_, libro) in enumerate(top_catalogo.iterrows(), start=1):
    print(f"{posicion}. {libro['title']}  ({libro['autor']})   puntuacion {libro['puntuacion']:.3f}")
    print(f"   {libro['synopsis']}")
    print()

1. Hijo de las sombras  (Sherrilyn Kenyon)   puntuacion 0.732
   Caillen Dagan ha aprendido a vivir en el Universo Ichidian casi como un espectro. Cuando lo detienen y están a punto de ejecutarlo, su ADN revela su verdadero origen. Entonces se ve obligado a adentrarse en un mundo aún más peligroso y despiadado que el de las sangrientas calles en las que se crió. Desideria persiste en su lucha por ganarse un lugar de honor en la guardia real de su madre. Pero cuando denuncia un complot para asesinar a la reina, se pondrá en el punto de mira de los conspiradores. Tanto Caillen como Desideria deberán aprender a luchar juntos, porque si fracasan, el mundo será un poco más desleal... y mucho más peligroso.

2. Esa horrible fortaleza  (C. S. Lewis)   puntuacion 0.724
   En su última aventura, a pesar de no salir de Thulcandra, la Tierra, el doctor Ransom se encuentra inmerso en un mundo de seres superiores que viven una frenética carrera científica. En este agresivo contexto dominado por la 

**Comentario del lector (Franco Renna):** me interesan *Esa horrible fortaleza*, *El maleficio* y *Perseguida por toda la ciudad*; *Hijo de las sombras* y *Un caso de urgencia* no.

Son **3 de 5** recomendaciones que le interesan al lector. Coincide con lo que mostró la prueba (precision@5 de 0,60), aunque este número no es una métrica: son libros nuevos que el lector juzgó después de ver la lista.

Estos comentarios son una revisión cualitativa sobre libros nuevos y no entran en las métricas de
prueba. Una puntuación alta es una estimación del modelo, no una garantía de que el libro le vaya a
gustar después de leerlo.

## Resumir las recomendaciones

Para cada libro recomendado se arma un resumen extractivo de dos oraciones: se divide la sinopsis en
oraciones, se representa cada una con TF-IDF, se calcula la similitud del coseno entre todas, y se
conservan las dos más **centrales** (las que más se parecen al resto) en su **orden original**.

In [74]:
def resumen_extractivo(texto, n_oraciones=2):
    # Las n oraciones mas centrales de la sinopsis, en el orden en que aparecen
    oraciones_texto = dividir(texto)
    if len(oraciones_texto) <= n_oraciones:
        return " ".join(oraciones_texto) if oraciones_texto else texto
    matriz = TfidfVectorizer().fit_transform(oraciones_texto)      # una fila por oracion de ESTA sinopsis
    similitudes = cosine_similarity(matriz)
    np.fill_diagonal(similitudes, 0)                               # una oracion no cuenta consigo misma
    centralidad = similitudes.sum(axis=1)
    elegidas = sorted(np.argsort(-centralidad)[:n_oraciones])      # orden original
    return " ".join(oraciones_texto[i] for i in elegidas)

for _, libro in top_catalogo.iterrows():
    print(f"{libro['title']}  ({len(dividir(libro['synopsis']))} oraciones)")
    print(f"   RESUMEN: {resumen_extractivo(libro['synopsis'])}")
    print()

Hijo de las sombras  (7 oraciones)
   RESUMEN: Entonces se ve obligado a adentrarse en un mundo aún más peligroso y despiadado que el de las sangrientas calles en las que se crió. Desideria persiste en su lucha por ganarse un lugar de honor en la guardia real de su madre.

Esa horrible fortaleza  (3 oraciones)
   RESUMEN: En su última aventura, a pesar de no salir de Thulcandra, la Tierra, el doctor Ransom se encuentra inmerso en un mundo de seres superiores que viven una frenética carrera científica. Su búsqueda de la verdad es un apasionante viaje lleno de intriga y suspense.

Un caso de urgencia  (4 oraciones)
   RESUMEN: Su amigo, el forense John Berry, está dispuesto a probar su inocencia, pero en el curso de su investigación chocará con el desprecio de la alta sociedad bostoniana, la cólera de la policía y la venganza de los bajos fondos. Y es así porque la trágica muerte de Karen Randall no es más que la punta del iceberg: un turbio laberinto de escándalos e hipocresías relacion

**Un resumen que funcionó: *El maleficio*.** De seis oraciones conserva la del secuestro de los
hermanos por la Bruja y la de los poderes que descubre Raquel. Con eso se entiende de qué trata el
libro: niños llevados a otro mundo y una protagonista con poderes que se enfrenta a la villana.

**Un resumen que perdió información: *Hijo de las sombras*.** De siete oraciones elige la cuarta
y la quinta: *"Entonces se ve obligado a adentrarse en un mundo aún más peligroso..."* y
*"Desideria persiste en su lucha..."*. Se pierde la primera oración, que presenta al protagonista
(Caillen Dagan), y la que cuenta el conflicto (el complot para asesinar a la reina). El resumen
empieza con un "Entonces" que remite a algo que no está y nombra a Desideria sin decir quién es.
Es una limitación del método: la centralidad mide qué oraciones comparten más palabras con el
resto, y la oración que presenta a un personaje o el giro de la trama suelen tener vocabulario
propio, así que quedan como poco centrales aunque sean las más importantes.

## Describir el perfil del lector

Con spaCy se extraen las frases sustantivas (`noun_chunks`) y las entidades nombradas de las sinopsis
de entrenamiento, por separado para los libros con `interes=1` y con `interes=0`. Las frases se agrupan
por el **lema de su núcleo**, así "los dragones" y "un dragón" cuentan como la misma idea. Se descartan
las frases cuyo núcleo es un pronombre, que no describen temas.

In [75]:
# El nlp del Paso 2 tiene el lematizador desactivado; aca hace falta, asi que se carga completo
nlp_perfil = spacy.load("es_core_news_lg")

def perfil_de(sinopsis):
    # Cuenta nucleos de frases sustantivas (por lema) y entidades (por texto y tipo)
    nucleos, entidades = Counter(), Counter()
    for doc in nlp_perfil.pipe(sinopsis, batch_size=20):
        for frase in doc.noun_chunks:
            if frase.root.pos_ in ("NOUN", "PROPN"):
                nucleos[frase.root.lemma_.lower()] += 1
        for e in doc.ents:
            entidades[f"{e.text} ({e.label_})"] += 1
    return nucleos, entidades

nucleos_1, entidades_1 = perfil_de(entrenamiento.loc[entrenamiento["interes"] == 1, "synopsis"].tolist())
nucleos_0, entidades_0 = perfil_de(entrenamiento.loc[entrenamiento["interes"] == 0, "synopsis"].tolist())

def tabla_top(conteo_1, conteo_0, titulo):
    # Las diez mas frecuentes de cada grupo, lado a lado
    top_1 = [f"{k} ({v})" for k, v in conteo_1.most_common(10)]
    top_0 = [f"{k} ({v})" for k, v in conteo_0.most_common(10)]
    print(titulo)
    print(pd.DataFrame({"le interesan (24 libros)": top_1, "no le interesan (56 libros)": top_0},
                       index=range(1, 11)).to_string())
    print()

tabla_top(nucleos_1, nucleos_0, "Nucleos de frases sustantivas (lema)")
tabla_top(entidades_1, entidades_0, "Entidades nombradas")

Nucleos de frases sustantivas (lema)
   le interesan (24 libros) no le interesan (56 libros)
1                libro (12)                  libro (25)
2                 vida (11)                 guerra (23)
3                mundo (11)                   obra (23)
4                 siglo (6)               historia (22)
5                   día (6)                    año (20)
6               ciencia (6)                   vida (20)
7                futuro (5)                  mundo (17)
8               persona (5)                  siglo (14)
9                 parte (5)                 hombre (14)
10         inteligencia (5)                  parte (14)

Entidades nombradas
     le interesan (24 libros)                                no le interesan (56 libros)
1            Kahlan (PER) (3)                                          España (LOC) (11)
2                IA (ORG) (3)                                          Javier (PER) (11)
3            Julián (PER) (3)                              

**Comentario del lector (Franco Renna):** sí, la lista me describe, aunque no entendía a qué se referían "persona" y "hombre".

**Lectura.** Las dos listas comparten núcleos genéricos (*libro*, *vida*, *mundo*, *siglo*,
*parte*), que vienen del lenguaje de las sinopsis y no del tema. Lo que las distingue está más
abajo. Entre los libros que le interesan aparecen **ciencia, futuro, inteligencia y persona**;
entre los que no, **guerra, obra, historia, año y hombre**. Las entidades van en el mismo sentido:
en los rechazados dominan España, Sevilla, Averroes, Pitágoras y títulos de ensayos sobre la
Antigüedad, y en los que le interesan, personajes de ficción (Kahlan, de la saga *La espada de la
verdad*) y la IA.

**De dónde salen "persona" y "hombre".** No son temas sino palabras generales, y su frecuencia refleja el tipo de libro. *Persona* aparece en 4 de los libros que le interesan, todos sobre comportamiento humano: *Hábitos para ser millonario* ("¿por qué algunas personas son más exitosas que otras?"), *Inteligencias múltiples*, *El lobo de Wall Street* y *Asesinos múltiples*. *Hombre* aparece en 11 de los que no le interesan, casi todos de historia, aventuras de época o literatura clásica (*Los mercenarios*, *La aventura de los conquistadores*, *Bímini*, la antología de cuentos rusos), donde se usa como "el hombre" genérico. La lista acierta en el tipo de libro, pero con una palabra que por sí sola no lo explica: es una limitación de contar núcleos de frases sueltos.

Dos advertencias. Muchas entidades aparecen dos o tres veces porque se repiten dentro de **una
misma sinopsis**, así que describen libros puntuales más que gustos. Y spaCy comete errores en
estas listas: etiqueta "IA" como organización y "Atayde" como miscelánea. Los núcleos de las
frases sustantivas describen mejor el perfil que las entidades.

## La función `recomendar`

Junta todo lo anterior: puntúa los candidatos con el clasificador de embeddings ya entrenado y
devuelve los `n` primeros con su identificador, título, puntuación y resumen de dos oraciones.

In [76]:
def recomendar(n=5):
    # Los n libros no valorados con mayor probabilidad de interes=1, segun el modelo B del Paso 5
    elegidos = candidatos.sort_values("puntuacion", ascending=False).head(n)
    return pd.DataFrame({
        "slug": elegidos["slug"].values,
        "title": elegidos["title"].values,
        "puntuacion": elegidos["puntuacion"].round(3).values,
        "resumen": [resumen_extractivo(s) for s in elegidos["synopsis"]],
    })

recomendaciones = recomendar(5)
print(recomendaciones[["slug", "title", "puntuacion"]].to_string(index=False))
print()
print("filas devueltas:", len(recomendaciones))
print("ninguna esta en preferencias.csv:", not recomendaciones["slug"].isin(preferencias["slug"]).any())
print("ninguna es otra version de un libro visto:",
      not recomendaciones["slug"].isin({s for s, _, _ in otras_versiones}).any())

                         slug                         title  puntuacion
          hijo-de-las-sombras           Hijo de las sombras       0.732
       esa-horrible-fortaleza        Esa horrible fortaleza       0.724
          un-caso-de-urgencia           Un caso de urgencia       0.720
                 el-maleficio                  El maleficio       0.706
perseguida-por-toda-la-ciudad Perseguida por toda la ciudad       0.701

filas devueltas: 5
ninguna esta en preferencias.csv: True
ninguna es otra version de un libro visto: True
